# Research notebook
Run the bootstrap cell first. Review the experiment parameters and data paths before executing the remaining cells. Outputs are intentionally cleared for version control.


In [ ]:
from pathlib import Path
import os
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))
Path("runs/notebooks").mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# MODULE 1
# STATE GRAPH - BUILD SNAPSHOT CACHE
# features:
#   - per-level raw LOB features: askp, asks, bidp, bids
#   - global semantic features: mid_price, spread,
#                               depth_ask_total, depth_bid_total,
#                               global_imbalance
# no lag, no differencing
# ============================================================

import os
import gc
import json
import warnings
warnings.filterwarnings("ignore")

import cudf
import cupy as cp
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "LOBSTER_SampleFile_AAPL_2012-06-21_10/AAPL_2012-06-21_34200000_57600000_orderbook_10.csv"

NUM_LEVELS = 10
NROWS = None

USE_FLOAT32 = True
EPS = 1e-8

OUTDIR = "./glasso_cache_v4"
os.makedirs(OUTDIR, exist_ok=True)

# Keep the SAME filenames as the original MODULE 1
X_RAW_NPY_PATH = os.path.join(OUTDIR, "lob_state_Xraw.npy")
XSTD_NPY_PATH = os.path.join(OUTDIR, "lob_state_Xstd.npy")
FEATURES_JSON_PATH = os.path.join(OUTDIR, "lob_state_feature_names.json")
SCALER_STATS_NPZ_PATH = os.path.join(OUTDIR, "lob_state_scaler_stats.npz")

EMP_COV_NPY_PATH = os.path.join(OUTDIR, "lob_state_empirical_covariance.npy")
EMP_CORR_NPY_PATH = os.path.join(OUTDIR, "lob_state_empirical_correlation.npy")

EMP_COV_CSV_PATH = os.path.join(OUTDIR, "lob_state_empirical_covariance.csv")
EMP_CORR_CSV_PATH = os.path.join(OUTDIR, "lob_state_empirical_correlation.csv")

META_JSON_PATH = os.path.join(OUTDIR, "lob_state_cache_metadata.json")


# ============================================================
# UTILS
# ============================================================

def clear_gpu_memory():
    """
    Release CuPy memory pools and trigger Python garbage collection.
    """
    try:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass
    gc.collect()


def empirical_covariance_from_standardized_data(X_std: np.ndarray) -> np.ndarray:
    """
    Compute empirical covariance from already standardized data.

    Parameters
    ----------
    X_std : np.ndarray
        Standardized design matrix of shape (n_samples, n_features).

    Returns
    -------
    np.ndarray
        Empirical covariance matrix of shape (n_features, n_features).
    """
    n_samples = X_std.shape[0]
    if n_samples < 2:
        raise ValueError("Need at least 2 samples to compute covariance.")

    X_std64 = X_std.astype(np.float64, copy=False)
    cov = (X_std64.T @ X_std64) / (n_samples - 1)
    return cov


def covariance_to_correlation(cov: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Convert a covariance matrix into a correlation matrix.

    Parameters
    ----------
    cov : np.ndarray
        Covariance matrix.
    eps : float
        Small constant to avoid division by zero.

    Returns
    -------
    np.ndarray
        Correlation matrix.
    """
    d = np.sqrt(np.clip(np.diag(cov), eps, None))
    corr = cov / np.outer(d, d)
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def build_expected_feature_names(num_levels: int) -> list[str]:
    """
    Build the expected ordered list of feature names.

    Order:
      1. per-level raw features
      2. global semantic features
    """
    feature_names = []

    for i in range(num_levels):
        feature_names.extend([
            f"askp_{i}",
            f"asks_{i}",
            f"bidp_{i}",
            f"bids_{i}",
        ])

    feature_names.extend([
        "mid_price",
        "spread",
        "depth_ask_total",
        "depth_bid_total",
        "global_imbalance",
    ])

    return feature_names


# ============================================================
# LOAD RAW LOBSTER CSV
# LOBSTER order per level:
# ask_price, ask_size, bid_price, bid_size
# repeated for each level
# ============================================================

print("Loading raw LOB CSV on GPU with cuDF...")

if NROWS is not None:
    df_raw = cudf.read_csv(CSV_PATH, header=None, nrows=NROWS)
else:
    df_raw = cudf.read_csv(CSV_PATH, header=None)

print(f"Loaded shape: {df_raw.shape}")


# ============================================================
# BASIC VALIDATION
# ============================================================

expected_num_columns = 4 * NUM_LEVELS
actual_num_columns = df_raw.shape[1]

if actual_num_columns < expected_num_columns:
    raise ValueError(
        f"Input CSV has only {actual_num_columns} columns, "
        f"but {expected_num_columns} are required for NUM_LEVELS={NUM_LEVELS}."
    )

print("\nInput validation passed.")
print(f"Expected minimum columns: {expected_num_columns}")
print(f"Actual columns found     : {actual_num_columns}")


# ============================================================
# BUILD SNAPSHOT STATE FEATURES
# ============================================================

print("\nBuilding snapshot state features...")

dtype_str = "float32" if USE_FLOAT32 else "float64"
feature_dict = {}

ask_size_cols = []
bid_size_cols = []

best_ask = None
best_bid = None

for i in range(NUM_LEVELS):
    ask_p_col = 4 * i + 0
    ask_s_col = 4 * i + 1
    bid_p_col = 4 * i + 2
    bid_s_col = 4 * i + 3

    ask_p = df_raw.iloc[:, ask_p_col].astype(dtype_str)
    ask_s = df_raw.iloc[:, ask_s_col].astype(dtype_str)
    bid_p = df_raw.iloc[:, bid_p_col].astype(dtype_str)
    bid_s = df_raw.iloc[:, bid_s_col].astype(dtype_str)

    feature_dict[f"askp_{i}"] = ask_p
    feature_dict[f"asks_{i}"] = ask_s
    feature_dict[f"bidp_{i}"] = bid_p
    feature_dict[f"bids_{i}"] = bid_s

    ask_size_cols.append(ask_s)
    bid_size_cols.append(bid_s)

    if i == 0:
        best_ask = ask_p
        best_bid = bid_p

if best_ask is None or best_bid is None:
    raise ValueError("Failed to build best ask / best bid from level 0.")

depth_ask_total = ask_size_cols[0]
for col in ask_size_cols[1:]:
    depth_ask_total = depth_ask_total + col

depth_bid_total = bid_size_cols[0]
for col in bid_size_cols[1:]:
    depth_bid_total = depth_bid_total + col

mid_price = (best_ask + best_bid) / 2.0
spread = best_ask - best_bid
global_imbalance = (depth_bid_total - depth_ask_total) / (depth_bid_total + depth_ask_total + EPS)

feature_dict["mid_price"] = mid_price
feature_dict["spread"] = spread
feature_dict["depth_ask_total"] = depth_ask_total
feature_dict["depth_bid_total"] = depth_bid_total
feature_dict["global_imbalance"] = global_imbalance

df_state = cudf.DataFrame(feature_dict)

feature_names = list(df_state.columns)
expected_feature_names = build_expected_feature_names(NUM_LEVELS)

if feature_names != expected_feature_names:
    raise ValueError(
        "State feature order mismatch.\n"
        f"Expected: {expected_feature_names}\n"
        f"Found:    {feature_names}"
    )

n_samples = len(df_state)
n_features = len(feature_names)

print(f"df_state shape: {df_state.shape}")
print(f"n_samples = {n_samples}")
print(f"n_features = {n_features}")
print("Feature names:")
print(feature_names)


# ============================================================
# CHECK MISSING / INVALID VALUES
# ============================================================

print("\nChecking for missing or invalid values...")

nan_counts = df_state.isnull().sum().to_pandas()
total_nan = int(nan_counts.sum())

print(f"Total NaN values in df_state: {total_nan}")

if total_nan > 0:
    raise ValueError(
        "NaN values detected in snapshot state features. "
        "Please inspect the raw data or add an explicit cleaning step."
    )


# ============================================================
# MOVE TO CPU
# ============================================================

print("\nConverting snapshot dataframe to CPU numpy array...")

dtype = np.float32 if USE_FLOAT32 else np.float64
X_raw = df_state.to_pandas().to_numpy(dtype=dtype, copy=True)

del df_state
del df_raw
clear_gpu_memory()

print(f"X_raw shape on CPU: {X_raw.shape}, dtype={X_raw.dtype}")


# ============================================================
# SAVE RAW MATRIX
# ============================================================

np.save(X_RAW_NPY_PATH, X_raw)
print(f"Saved raw matrix            -> {os.path.abspath(X_RAW_NPY_PATH)}")


# ============================================================
# STANDARDIZE
# ============================================================

print("\nStandardizing features with StandardScaler...")

scaler = StandardScaler(with_mean=True, with_std=True)
X_std = scaler.fit_transform(X_raw)
X_std = X_std.astype(dtype, copy=False)

np.save(XSTD_NPY_PATH, X_std)

np.savez(
    SCALER_STATS_NPZ_PATH,
    mean_=scaler.mean_.astype(np.float64),
    scale_=scaler.scale_.astype(np.float64),
    var_=scaler.var_.astype(np.float64),
)

with open(FEATURES_JSON_PATH, "w") as f:
    json.dump(feature_names, f, indent=2)

print(f"Saved standardized matrix   -> {os.path.abspath(XSTD_NPY_PATH)}")
print(f"Saved scaler stats          -> {os.path.abspath(SCALER_STATS_NPZ_PATH)}")
print(f"Saved feature names         -> {os.path.abspath(FEATURES_JSON_PATH)}")


# ============================================================
# EMPIRICAL COVARIANCE / CORRELATION
# ============================================================

print("\nComputing empirical covariance and correlation...")

emp_cov = empirical_covariance_from_standardized_data(X_std)
emp_corr = covariance_to_correlation(emp_cov)

np.save(EMP_COV_NPY_PATH, emp_cov)
np.save(EMP_CORR_NPY_PATH, emp_corr)

emp_cov_df = pd.DataFrame(emp_cov, index=feature_names, columns=feature_names)
emp_corr_df = pd.DataFrame(emp_corr, index=feature_names, columns=feature_names)

emp_cov_df.to_csv(EMP_COV_CSV_PATH, index=True)
emp_corr_df.to_csv(EMP_CORR_CSV_PATH, index=True)

print(f"Saved covariance NPY        -> {os.path.abspath(EMP_COV_NPY_PATH)}")
print(f"Saved correlation NPY       -> {os.path.abspath(EMP_CORR_NPY_PATH)}")
print(f"Saved covariance CSV        -> {os.path.abspath(EMP_COV_CSV_PATH)}")
print(f"Saved correlation CSV       -> {os.path.abspath(EMP_CORR_CSV_PATH)}")


# ============================================================
# METADATA
# ============================================================

meta = {
    "module": "MODULE 1 - STATE GRAPH - BUILD SNAPSHOT CACHE",
    "csv_path": CSV_PATH,
    "num_levels": NUM_LEVELS,
    "nrows": NROWS,
    "use_float32": USE_FLOAT32,
    "eps": EPS,
    "n_samples": int(n_samples),
    "n_features": int(n_features),
    "feature_names": feature_names,
    "raw_matrix_npy": X_RAW_NPY_PATH,
    "xstd_npy": XSTD_NPY_PATH,
    "scaler_stats_npz": SCALER_STATS_NPZ_PATH,
    "feature_names_json": FEATURES_JSON_PATH,
    "emp_cov_npy": EMP_COV_NPY_PATH,
    "emp_corr_npy": EMP_CORR_NPY_PATH,
    "emp_cov_csv": EMP_COV_CSV_PATH,
    "emp_corr_csv": EMP_CORR_CSV_PATH,
    "semantic_features_added": [
        "mid_price",
        "spread",
        "depth_ask_total",
        "depth_bid_total",
        "global_imbalance",
    ],
}

with open(META_JSON_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata              -> {os.path.abspath(META_JSON_PATH)}")


# ============================================================
# QUICK CHECKS
# ============================================================

diag_mean_cov = float(np.mean(np.diag(emp_cov)))
diag_mean_corr = float(np.mean(np.diag(emp_corr)))
symmetry_error_cov = float(np.max(np.abs(emp_cov - emp_cov.T)))
symmetry_error_corr = float(np.max(np.abs(emp_corr - emp_corr.T)))

print("\nQuick checks:")
print(f"Covariance diagonal mean      : {diag_mean_cov:.6f}")
print(f"Correlation diagonal mean     : {diag_mean_corr:.6f}")
print(f"Max covariance symmetry error : {symmetry_error_cov:.6e}")
print(f"Max correlation symmetry error: {symmetry_error_corr:.6e}")

print("\nDONE: state snapshot cache created successfully.")

In [ ]:
# ============================================================
# MODULE 2
# STATE GRAPH - BUILD SPATIO-TEMPORAL LAGGED CACHE
# input : raw snapshot cache from MODULE 1
# output: lagged raw matrix + lagged standardized matrix
# no differencing
# ============================================================

import os
import gc
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIG
# ============================================================

STATE_CACHE_DIR = "./glasso_cache_v4"

X_RAW_NPY_PATH = os.path.join(STATE_CACHE_DIR, "lob_state_Xraw.npy")
FEATURES_JSON_PATH = os.path.join(STATE_CACHE_DIR, "lob_state_feature_names.json")

MAX_LAG = 20
USE_FLOAT32 = True

OUTDIR = "./glasso_cache_v4/lagged_cache"
os.makedirs(OUTDIR, exist_ok=True)

X_LAGGED_RAW_NPY_PATH = os.path.join(OUTDIR, "lob_state_lagged_Xraw.npy")
X_LAGGED_STD_NPY_PATH = os.path.join(OUTDIR, "lob_state_lagged_Xstd.npy")
LAGGED_FEATURES_JSON_PATH = os.path.join(OUTDIR, "lob_state_lagged_feature_names.json")
SCALER_STATS_NPZ_PATH = os.path.join(OUTDIR, "lob_state_lagged_scaler_stats.npz")

EMP_COV_NPY_PATH = os.path.join(OUTDIR, "lob_state_lagged_empirical_covariance.npy")
EMP_CORR_NPY_PATH = os.path.join(OUTDIR, "lob_state_lagged_empirical_correlation.npy")

EMP_COV_CSV_PATH = os.path.join(OUTDIR, "lob_state_lagged_empirical_covariance.csv")
EMP_CORR_CSV_PATH = os.path.join(OUTDIR, "lob_state_lagged_empirical_correlation.csv")

META_JSON_PATH = os.path.join(OUTDIR, "lob_state_lagged_cache_metadata.json")


# ============================================================
# UTILS
# ============================================================

def empirical_covariance_from_standardized_data(X_std: np.ndarray) -> np.ndarray:
    """
    Compute empirical covariance from already standardized data.

    Parameters
    ----------
    X_std : np.ndarray
        Standardized design matrix of shape (n_samples, n_features).

    Returns
    -------
    np.ndarray
        Empirical covariance matrix of shape (n_features, n_features).
    """
    n_samples = X_std.shape[0]
    if n_samples < 2:
        raise ValueError("Need at least 2 samples to compute covariance.")

    X_std64 = X_std.astype(np.float64, copy=False)
    cov = (X_std64.T @ X_std64) / (n_samples - 1)
    return cov


def covariance_to_correlation(cov: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Convert a covariance matrix into a correlation matrix.
    """
    d = np.sqrt(np.clip(np.diag(cov), eps, None))
    corr = cov / np.outer(d, d)
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def build_lagged_feature_names(base_feature_names: list[str], max_lag: int) -> list[str]:
    """
    Build ordered lagged feature names.

    Ordering convention:
        all lag_0 features first,
        then all lag_1 features,
        ...
        then all lag_K features.

    Example:
        askp_0_lag_0, asks_0_lag_0, ... ,
        global_imbalance_lag_0,
        askp_0_lag_1, asks_0_lag_1, ... ,
        global_imbalance_lag_1,
        ...
    """
    lagged_feature_names = []

    for lag in range(max_lag + 1):
        for feat in base_feature_names:
            lagged_feature_names.append(f"{feat}_lag_{lag}")

    return lagged_feature_names


def build_lagged_matrix(X_raw: np.ndarray, max_lag: int) -> np.ndarray:
    """
    Build the lagged spatio-temporal design matrix from the raw snapshot matrix.

    For each valid time index t >= max_lag, the output row is:

        [x_t, x_{t-1}, x_{t-2}, ..., x_{t-max_lag}]

    Parameters
    ----------
    X_raw : np.ndarray
        Raw snapshot matrix of shape (T, F).
    max_lag : int
        Maximum lag to include.

    Returns
    -------
    np.ndarray
        Lagged matrix of shape (T - max_lag, F * (max_lag + 1)).
    """
    T, F = X_raw.shape

    if max_lag < 0:
        raise ValueError("MAX_LAG must be >= 0.")

    if T <= max_lag:
        raise ValueError(
            f"Not enough rows to build lagged matrix: T={T}, MAX_LAG={max_lag}."
        )

    n_samples_out = T - max_lag
    lagged_blocks = []

    for lag in range(max_lag + 1):
        start = max_lag - lag
        end = T - lag
        block = X_raw[start:end]
        lagged_blocks.append(block)

    X_lagged = np.concatenate(lagged_blocks, axis=1)
    expected_shape = (n_samples_out, F * (max_lag + 1))

    if X_lagged.shape != expected_shape:
        raise ValueError(
            f"Lagged matrix shape mismatch. "
            f"Expected {expected_shape}, got {X_lagged.shape}."
        )

    return X_lagged


# ============================================================
# LOAD SNAPSHOT CACHE FROM MODULE 1
# ============================================================

print("Loading raw snapshot cache from MODULE 1...")

if not os.path.exists(X_RAW_NPY_PATH):
    raise FileNotFoundError(f"Missing raw snapshot matrix: {X_RAW_NPY_PATH}")

if not os.path.exists(FEATURES_JSON_PATH):
    raise FileNotFoundError(f"Missing base feature names: {FEATURES_JSON_PATH}")

X_raw = np.load(X_RAW_NPY_PATH)

with open(FEATURES_JSON_PATH, "r") as f:
    base_feature_names = json.load(f)

print(f"Loaded X_raw shape          : {X_raw.shape}")
print(f"Loaded number of features   : {len(base_feature_names)}")


# ============================================================
# BASIC VALIDATION
# ============================================================

if X_raw.ndim != 2:
    raise ValueError(f"X_raw must be 2D, got shape {X_raw.shape}")

n_samples_raw, n_features_raw = X_raw.shape

if n_features_raw != len(base_feature_names):
    raise ValueError(
        "Mismatch between X_raw columns and base feature names.\n"
        f"X_raw columns       : {n_features_raw}\n"
        f"Feature names count : {len(base_feature_names)}"
    )

if MAX_LAG < 0:
    raise ValueError("MAX_LAG must be >= 0.")

print("\nInput validation passed.")
print(f"n_samples_raw   = {n_samples_raw}")
print(f"n_features_raw  = {n_features_raw}")
print(f"MAX_LAG         = {MAX_LAG}")


# ============================================================
# BUILD LAGGED MATRIX
# Convention:
# row(t) = [x_t, x_{t-1}, ..., x_{t-MAX_LAG}]
# ============================================================

print("\nBuilding lagged spatio-temporal matrix...")

dtype = np.float32 if USE_FLOAT32 else np.float64
X_raw = X_raw.astype(dtype, copy=False)

X_lagged_raw = build_lagged_matrix(X_raw, MAX_LAG)
lagged_feature_names = build_lagged_feature_names(base_feature_names, MAX_LAG)

n_samples_lagged, n_features_lagged = X_lagged_raw.shape

if n_features_lagged != len(lagged_feature_names):
    raise ValueError(
        "Mismatch between lagged matrix columns and lagged feature names.\n"
        f"X_lagged columns      : {n_features_lagged}\n"
        f"Lagged names count    : {len(lagged_feature_names)}"
    )

print(f"X_lagged_raw shape      : {X_lagged_raw.shape}")
print(f"n_samples_lagged        : {n_samples_lagged}")
print(f"n_features_lagged       : {n_features_lagged}")
print("First 15 lagged feature names:")
print(lagged_feature_names[:15])


# ============================================================
# SAVE RAW LAGGED MATRIX
# ============================================================

np.save(X_LAGGED_RAW_NPY_PATH, X_lagged_raw)
with open(LAGGED_FEATURES_JSON_PATH, "w") as f:
    json.dump(lagged_feature_names, f, indent=2)

print(f"\nSaved raw lagged matrix     -> {os.path.abspath(X_LAGGED_RAW_NPY_PATH)}")
print(f"Saved lagged feature names  -> {os.path.abspath(LAGGED_FEATURES_JSON_PATH)}")


# ============================================================
# STANDARDIZE
# ============================================================

print("\nStandardizing lagged features with StandardScaler...")

scaler = StandardScaler(with_mean=True, with_std=True)
X_lagged_std = scaler.fit_transform(X_lagged_raw)
X_lagged_std = X_lagged_std.astype(dtype, copy=False)

np.save(X_LAGGED_STD_NPY_PATH, X_lagged_std)

np.savez(
    SCALER_STATS_NPZ_PATH,
    mean_=scaler.mean_.astype(np.float64),
    scale_=scaler.scale_.astype(np.float64),
    var_=scaler.var_.astype(np.float64),
)

print(f"Saved standardized matrix   -> {os.path.abspath(X_LAGGED_STD_NPY_PATH)}")
print(f"Saved scaler stats          -> {os.path.abspath(SCALER_STATS_NPZ_PATH)}")


# ============================================================
# EMPIRICAL COVARIANCE / CORRELATION
# ============================================================

print("\nComputing empirical covariance and correlation...")

emp_cov = empirical_covariance_from_standardized_data(X_lagged_std)
emp_corr = covariance_to_correlation(emp_cov)

np.save(EMP_COV_NPY_PATH, emp_cov)
np.save(EMP_CORR_NPY_PATH, emp_corr)

emp_cov_df = pd.DataFrame(emp_cov, index=lagged_feature_names, columns=lagged_feature_names)
emp_corr_df = pd.DataFrame(emp_corr, index=lagged_feature_names, columns=lagged_feature_names)

emp_cov_df.to_csv(EMP_COV_CSV_PATH, index=True)
emp_corr_df.to_csv(EMP_CORR_CSV_PATH, index=True)

print(f"Saved covariance NPY        -> {os.path.abspath(EMP_COV_NPY_PATH)}")
print(f"Saved correlation NPY       -> {os.path.abspath(EMP_CORR_NPY_PATH)}")
print(f"Saved covariance CSV        -> {os.path.abspath(EMP_COV_CSV_PATH)}")
print(f"Saved correlation CSV       -> {os.path.abspath(EMP_CORR_CSV_PATH)}")


# ============================================================
# METADATA
# ============================================================

meta = {
    "module": "MODULE 2 - STATE GRAPH - BUILD SPATIO-TEMPORAL LAGGED CACHE",
    "input_raw_snapshot_matrix": X_RAW_NPY_PATH,
    "input_base_feature_names": FEATURES_JSON_PATH,
    "max_lag": MAX_LAG,
    "use_float32": USE_FLOAT32,
    "n_samples_raw": int(n_samples_raw),
    "n_features_raw": int(n_features_raw),
    "n_samples_lagged": int(n_samples_lagged),
    "n_features_lagged": int(n_features_lagged),
    "base_feature_names": base_feature_names,
    "lagged_feature_names": lagged_feature_names,
    "x_lagged_raw_npy": X_LAGGED_RAW_NPY_PATH,
    "x_lagged_std_npy": X_LAGGED_STD_NPY_PATH,
    "scaler_stats_npz": SCALER_STATS_NPZ_PATH,
    "lagged_feature_names_json": LAGGED_FEATURES_JSON_PATH,
    "emp_cov_npy": EMP_COV_NPY_PATH,
    "emp_corr_npy": EMP_CORR_NPY_PATH,
    "emp_cov_csv": EMP_COV_CSV_PATH,
    "emp_corr_csv": EMP_CORR_CSV_PATH,
}

with open(META_JSON_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata              -> {os.path.abspath(META_JSON_PATH)}")


# ============================================================
# QUICK CHECKS
# ============================================================

diag_mean_cov = float(np.mean(np.diag(emp_cov)))
diag_mean_corr = float(np.mean(np.diag(emp_corr)))
symmetry_error_cov = float(np.max(np.abs(emp_cov - emp_cov.T)))
symmetry_error_corr = float(np.max(np.abs(emp_corr - emp_corr.T)))

print("\nQuick checks:")
print(f"Covariance diagonal mean      : {diag_mean_cov:.6f}")
print(f"Correlation diagonal mean     : {diag_mean_corr:.6f}")
print(f"Max covariance symmetry error : {symmetry_error_cov:.6e}")
print(f"Max correlation symmetry error: {symmetry_error_corr:.6e}")

print("\nDONE: lagged spatio-temporal cache created successfully.")

In [ ]:
# ============================================================
# MODULE 2B
# STATE GRAPH - PRUNE HIGHLY COLLINEAR LAGGED FEATURES
# input : lagged standardized cache from MODULE 2
# output: pruned lagged standardized matrix + pruned feature names
# ============================================================

import os
import gc
import json
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LAGGED_CACHE_DIR = "./glasso_cache_v4/lagged_cache"

X_LAGGED_STD_NPY_PATH = os.path.join(LAGGED_CACHE_DIR, "lob_state_lagged_Xstd.npy")
LAGGED_FEATURES_JSON_PATH = os.path.join(LAGGED_CACHE_DIR, "lob_state_lagged_feature_names.json")

THRESHOLD = 0.999

OUTDIR = "./glasso_cache_v4/lagged_cache_pruned"
os.makedirs(OUTDIR, exist_ok=True)

X_PRUNED_NPY_PATH = os.path.join(OUTDIR, "lob_state_lagged_Xstd_pruned.npy")
FEATURES_PRUNED_JSON_PATH = os.path.join(OUTDIR, "lob_state_lagged_feature_names_pruned.json")

DROPPED_FEATURES_CSV_PATH = os.path.join(OUTDIR, "lob_state_lagged_dropped_features.csv")
HIGH_CORR_PAIRS_CSV_PATH = os.path.join(OUTDIR, "lob_state_lagged_high_corr_pairs_pruned_step.csv")
META_JSON_PATH = os.path.join(OUTDIR, "lob_state_lagged_pruned_metadata.json")


# ============================================================
# UTILS
# ============================================================

def clear_memory():
    """
    Trigger Python garbage collection.
    """
    gc.collect()


def parse_feature_name(feature_name: str) -> dict:
    """
    Parse lagged feature names.

    Supported formats:

    1) Per-level features:
        askp_0_lag_0
        asks_3_lag_2
        bidp_7_lag_1
        bids_9_lag_0

    2) Global semantic features:
        mid_price_lag_0
        spread_lag_1
        depth_ask_total_lag_2
        depth_bid_total_lag_3
        global_imbalance_lag_0
    """

    # --------------------------------------------------------
    # Case 1: per-level raw features
    # --------------------------------------------------------
    pattern_level = r"^(askp|asks|bidp|bids)_(\d+)_lag_(\d+)$"
    match_level = re.match(pattern_level, feature_name)

    if match_level is not None:
        return {
            "feature_type": "level",
            "base_feature": match_level.group(1),
            "level": int(match_level.group(2)),
            "lag": int(match_level.group(3)),
        }

    # --------------------------------------------------------
    # Case 2: global semantic features
    # --------------------------------------------------------
    pattern_global = r"^(mid_price|spread|depth_ask_total|depth_bid_total|global_imbalance)_lag_(\d+)$"
    match_global = re.match(pattern_global, feature_name)

    if match_global is not None:
        return {
            "feature_type": "global",
            "base_feature": match_global.group(1),
            "level": -1,   # no level for global features
            "lag": int(match_global.group(2)),
        }

    raise ValueError(f"Unsupported feature name format: {feature_name}")


def feature_priority(feature_name: str) -> tuple:
    """
    Priority rule used when two features are almost duplicates.

    Lower tuple = higher priority = more likely to be kept.

    Priority:
    1. Smaller lag first (keep more recent information)
    2. Global features before per-level features if same lag
    3. Smaller level first for level-based features
    4. Lexicographic order of base feature
    """
    info = parse_feature_name(feature_name)

    feature_type_rank = 0 if info["feature_type"] == "global" else 1

    return (
        info["lag"],
        feature_type_rank,
        info["level"],
        info["base_feature"],
    )


def covariance_to_correlation(cov: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Convert covariance to correlation matrix.
    """
    d = np.sqrt(np.clip(np.diag(cov), eps, None))
    corr = cov / np.outer(d, d)
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def empirical_covariance_from_standardized_data(X_std: np.ndarray) -> np.ndarray:
    """
    Compute empirical covariance from already standardized data.
    """
    n_samples = X_std.shape[0]
    if n_samples < 2:
        raise ValueError("Need at least 2 samples to compute covariance.")

    X_std64 = X_std.astype(np.float64, copy=False)
    cov = (X_std64.T @ X_std64) / (n_samples - 1)
    return cov


# ============================================================
# LOAD LAGGED STANDARDIZED CACHE
# ============================================================

print("Loading lagged standardized cache from MODULE 2...")

if not os.path.exists(X_LAGGED_STD_NPY_PATH):
    raise FileNotFoundError(f"Missing lagged standardized matrix: {X_LAGGED_STD_NPY_PATH}")

if not os.path.exists(LAGGED_FEATURES_JSON_PATH):
    raise FileNotFoundError(f"Missing lagged feature names: {LAGGED_FEATURES_JSON_PATH}")

X = np.load(X_LAGGED_STD_NPY_PATH)

with open(LAGGED_FEATURES_JSON_PATH, "r") as f:
    feature_names = json.load(f)

print(f"Loaded X shape              : {X.shape}")
print(f"Loaded number of features   : {len(feature_names)}")


# ============================================================
# BASIC VALIDATION
# ============================================================

if X.ndim != 2:
    raise ValueError(f"X must be 2D, got shape {X.shape}")

n_samples, n_features = X.shape

if n_features != len(feature_names):
    raise ValueError(
        "Mismatch between X columns and feature names.\n"
        f"X columns            : {n_features}\n"
        f"Feature names count  : {len(feature_names)}"
    )

if THRESHOLD <= 0 or THRESHOLD >= 1:
    raise ValueError("THRESHOLD must satisfy 0 < THRESHOLD < 1.")

print("\nInput validation passed.")
print(f"n_samples   = {n_samples}")
print(f"n_features  = {n_features}")
print(f"THRESHOLD   = {THRESHOLD}")


# ============================================================
# COMPUTE CORRELATION MATRIX
# ============================================================

print("\nComputing lagged correlation matrix...")

cov = empirical_covariance_from_standardized_data(X)
corr = covariance_to_correlation(cov)

print("Correlation matrix computed successfully.")


# ============================================================
# FIND HIGH-CORRELATION PAIRS
# ============================================================

print("\nSearching for highly correlated pairs...")

high_corr_rows = []

for i in range(n_features):
    for j in range(i + 1, n_features):
        c = float(corr[i, j])
        if abs(c) >= THRESHOLD:
            high_corr_rows.append({
                "feature_i": feature_names[i],
                "feature_j": feature_names[j],
                "correlation": c,
                "abs_correlation": abs(c),
            })

high_corr_df = pd.DataFrame(high_corr_rows)

if not high_corr_df.empty:
    high_corr_df.sort_values(
        by=["abs_correlation", "feature_i", "feature_j"],
        ascending=[False, True, True],
        inplace=True,
    )
    high_corr_df.reset_index(drop=True, inplace=True)

high_corr_df.to_csv(HIGH_CORR_PAIRS_CSV_PATH, index=False)

print(f"Found high-correlation pairs : {len(high_corr_df)}")
print(f"Saved pair list              -> {os.path.abspath(HIGH_CORR_PAIRS_CSV_PATH)}")


# ============================================================
# GREEDY PRUNING
# Keep the higher-priority feature among highly correlated pairs
# ============================================================

print("\nPruning highly collinear features...")

keep_mask = np.ones(n_features, dtype=bool)
dropped_rows = []

for i in range(n_features):
    if not keep_mask[i]:
        continue

    for j in range(i + 1, n_features):
        if not keep_mask[j]:
            continue

        c = float(corr[i, j])
        if abs(c) < THRESHOLD:
            continue

        fi = feature_names[i]
        fj = feature_names[j]

        pri_i = feature_priority(fi)
        pri_j = feature_priority(fj)

        if pri_i <= pri_j:
            keep_mask[j] = False
            dropped_rows.append({
                "kept_feature": fi,
                "dropped_feature": fj,
                "correlation": c,
                "abs_correlation": abs(c),
                "kept_priority": str(pri_i),
                "dropped_priority": str(pri_j),
            })
        else:
            keep_mask[i] = False
            dropped_rows.append({
                "kept_feature": fj,
                "dropped_feature": fi,
                "correlation": c,
                "abs_correlation": abs(c),
                "kept_priority": str(pri_j),
                "dropped_priority": str(pri_i),
            })
            break

X_pruned = X[:, keep_mask]
feature_names_pruned = [f for f, keep in zip(feature_names, keep_mask) if keep]

dropped_df = pd.DataFrame(dropped_rows)
if not dropped_df.empty:
    dropped_df.sort_values(
        by=["abs_correlation", "kept_feature", "dropped_feature"],
        ascending=[False, True, True],
        inplace=True,
    )
    dropped_df.reset_index(drop=True, inplace=True)

dropped_df.to_csv(DROPPED_FEATURES_CSV_PATH, index=False)

print(f"Original features           : {n_features}")
print(f"Remaining features          : {len(feature_names_pruned)}")
print(f"Dropped features            : {n_features - len(feature_names_pruned)}")
print(f"Saved dropped-feature log   -> {os.path.abspath(DROPPED_FEATURES_CSV_PATH)}")


# ============================================================
# SAVE PRUNED OUTPUTS
# ============================================================

np.save(X_PRUNED_NPY_PATH, X_pruned)

with open(FEATURES_PRUNED_JSON_PATH, "w") as f:
    json.dump(feature_names_pruned, f, indent=2)

print(f"\nSaved pruned matrix         -> {os.path.abspath(X_PRUNED_NPY_PATH)}")
print(f"Saved pruned feature names  -> {os.path.abspath(FEATURES_PRUNED_JSON_PATH)}")


# ============================================================
# POST-PRUNING CHECK
# ============================================================

print("\nRunning post-pruning correlation check...")

cov_pruned = empirical_covariance_from_standardized_data(X_pruned)
corr_pruned = covariance_to_correlation(cov_pruned)

remaining_high_corr = 0
n_pruned_features = corr_pruned.shape[0]

for i in range(n_pruned_features):
    for j in range(i + 1, n_pruned_features):
        if abs(corr_pruned[i, j]) >= THRESHOLD:
            remaining_high_corr += 1

print(f"Remaining high-corr pairs   : {remaining_high_corr}")


# ============================================================
# METADATA
# ============================================================

meta = {
    "module": "MODULE 2B - STATE GRAPH - PRUNE HIGHLY COLLINEAR LAGGED FEATURES",
    "input_x_lagged_std_npy": X_LAGGED_STD_NPY_PATH,
    "input_lagged_feature_names_json": LAGGED_FEATURES_JSON_PATH,
    "threshold": float(THRESHOLD),
    "n_samples": int(n_samples),
    "n_features_original": int(n_features),
    "n_features_pruned": int(len(feature_names_pruned)),
    "n_features_dropped": int(n_features - len(feature_names_pruned)),
    "n_high_corr_pairs_before": int(len(high_corr_df)),
    "n_high_corr_pairs_after": int(remaining_high_corr),
    "x_pruned_npy": X_PRUNED_NPY_PATH,
    "feature_names_pruned_json": FEATURES_PRUNED_JSON_PATH,
    "dropped_features_csv": DROPPED_FEATURES_CSV_PATH,
    "high_corr_pairs_csv": HIGH_CORR_PAIRS_CSV_PATH,
}

with open(META_JSON_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata              -> {os.path.abspath(META_JSON_PATH)}")


# ============================================================
# QUICK CHECKS
# ============================================================

print("\nQuick checks:")
print(f"Threshold                    : {THRESHOLD}")
print(f"Original features            : {n_features}")
print(f"Remaining features           : {len(feature_names_pruned)}")
print(f"Dropped features             : {n_features - len(feature_names_pruned)}")
print(f"High-corr pairs before       : {len(high_corr_df)}")
print(f"High-corr pairs after        : {remaining_high_corr}")

print("\nDONE: lagged feature pruning completed successfully.")

In [ ]:
# ============================================================
# MODULE 3
# STATE GRAPH - GRAPHICAL LASSO CV FOR ALPHA SELECTION
# input : pruned lagged standardized cache from MODULE 2B
# output: best alpha + CV results
# ============================================================

import os
import gc
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.covariance import GraphicalLasso
from sklearn.model_selection import KFold


# ============================================================
# CONFIG
# ============================================================

LAGGED_CACHE_DIR = "./glasso_cache_v4/lagged_cache_pruned"

X_LAGGED_STD_NPY_PATH = os.path.join(LAGGED_CACHE_DIR, "lob_state_lagged_Xstd_pruned.npy")
LAGGED_FEATURES_JSON_PATH = os.path.join(LAGGED_CACHE_DIR, "lob_state_lagged_feature_names_pruned.json")

ALPHA_GRID = np.array([0.005, 0.007, 0.01, 0.05, 0.07, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0], dtype=float)

N_SPLITS = 5
MIN_SUCCESS_FOLDS = 4
SHUFFLE = False
RANDOM_STATE = 42

MAX_ITER = 200
TOL = 1e-4

EPS = 1e-8
USE_FLOAT32 = True

OUTDIR = "./glasso_cache_v4/state_graph_glasso_cv"
os.makedirs(OUTDIR, exist_ok=True)

CV_RESULTS_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_cv_results.csv")
CV_RESULTS_JSON_PATH = os.path.join(OUTDIR, "lob_state_glasso_cv_results.json")
BEST_ALPHA_JSON_PATH = os.path.join(OUTDIR, "lob_state_glasso_best_alpha.json")
META_JSON_PATH = os.path.join(OUTDIR, "lob_state_glasso_cv_metadata.json")


# ============================================================
# UTILS
# ============================================================

def clear_memory():
    """
    Trigger Python garbage collection.
    """
    gc.collect()


def empirical_covariance(X: np.ndarray) -> np.ndarray:
    """
    Compute empirical covariance matrix from a 2D data matrix.
    """
    n_samples = X.shape[0]
    if n_samples < 2:
        raise ValueError("Need at least 2 samples to compute covariance.")

    X64 = X.astype(np.float64, copy=False)
    X_centered = X64 - X64.mean(axis=0, keepdims=True)
    cov = (X_centered.T @ X_centered) / (n_samples - 1)
    return cov


def safe_logdet(A: np.ndarray, eps: float = 1e-12) -> float:
    """
    Compute a numerically safe log-determinant.
    """
    A64 = A.astype(np.float64, copy=False)
    sign, logdet = np.linalg.slogdet(A64)

    if sign > 0:
        return float(logdet)

    A_jitter = A64 + eps * np.eye(A64.shape[0], dtype=np.float64)
    sign_j, logdet_j = np.linalg.slogdet(A_jitter)

    if sign_j <= 0:
        raise ValueError("Matrix is not positive definite even after jitter.")

    return float(logdet_j)


def gaussian_neg_log_likelihood_from_emp_cov(
    emp_cov_val: np.ndarray,
    precision_hat: np.ndarray,
    eps: float = 1e-12,
) -> float:
    """
    Compute the Gaussian negative log-likelihood (up to constants)
    on validation empirical covariance:

        NLL = trace(S_val @ Theta_hat) - logdet(Theta_hat)

    Lower is better.
    """
    S = emp_cov_val.astype(np.float64, copy=False)
    Theta = precision_hat.astype(np.float64, copy=False)

    trace_term = float(np.trace(S @ Theta))
    logdet_term = safe_logdet(Theta, eps=eps)

    return trace_term - logdet_term


def count_offdiag_nonzero_entries(
    precision: np.ndarray,
    threshold: float = 1e-12
) -> int:
    """
    Count non-zero off-diagonal entries in the precision matrix.
    """
    P = precision.copy()
    np.fill_diagonal(P, 0.0)
    return int(np.sum(np.abs(P) > threshold))


# ============================================================
# LOAD PRUNED LAGGED STANDARDIZED CACHE
# ============================================================

print("Loading pruned lagged standardized cache from MODULE 2B...")

if not os.path.exists(X_LAGGED_STD_NPY_PATH):
    raise FileNotFoundError(f"Missing pruned lagged standardized matrix: {X_LAGGED_STD_NPY_PATH}")

if not os.path.exists(LAGGED_FEATURES_JSON_PATH):
    raise FileNotFoundError(f"Missing pruned lagged feature names: {LAGGED_FEATURES_JSON_PATH}")

X = np.load(X_LAGGED_STD_NPY_PATH)

with open(LAGGED_FEATURES_JSON_PATH, "r") as f:
    lagged_feature_names = json.load(f)

print(f"Loaded X shape              : {X.shape}")
print(f"Loaded number of features   : {len(lagged_feature_names)}")


# ============================================================
# BASIC VALIDATION
# ============================================================

if X.ndim != 2:
    raise ValueError(f"X must be 2D, got shape {X.shape}")

n_samples, n_features = X.shape

if n_features != len(lagged_feature_names):
    raise ValueError(
        "Mismatch between X columns and lagged feature names.\n"
        f"X columns            : {n_features}\n"
        f"Feature names count  : {len(lagged_feature_names)}"
    )

if N_SPLITS < 2:
    raise ValueError("N_SPLITS must be >= 2.")

if MIN_SUCCESS_FOLDS < 1 or MIN_SUCCESS_FOLDS > N_SPLITS:
    raise ValueError("MIN_SUCCESS_FOLDS must satisfy 1 <= MIN_SUCCESS_FOLDS <= N_SPLITS.")

if n_samples <= N_SPLITS:
    raise ValueError(
        f"Not enough samples for cross-validation: n_samples={n_samples}, N_SPLITS={N_SPLITS}"
    )

dtype = np.float32 if USE_FLOAT32 else np.float64
X = X.astype(dtype, copy=False)

print("\nInput validation passed.")
print(f"n_samples         = {n_samples}")
print(f"n_features        = {n_features}")
print(f"N_SPLITS          = {N_SPLITS}")
print(f"MIN_SUCCESS_FOLDS = {MIN_SUCCESS_FOLDS}")
print(f"MAX_ITER          = {MAX_ITER}")
print(f"TOL               = {TOL}")
print(f"Alpha grid        = {ALPHA_GRID.tolist()}")


# ============================================================
# CROSS-VALIDATION SETUP
# ============================================================

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE if SHUFFLE else None
)

cv_rows = []

print("\nStarting Graphical Lasso cross-validation...")


# ============================================================
# CROSS-VALIDATION LOOP
# Metric:
# validation Gaussian negative log-likelihood
# Lower is better
# ============================================================

for alpha in ALPHA_GRID:
    alpha = float(alpha)
    print(f"\nEvaluating alpha = {alpha:.8f}")

    fold_scores = []
    fold_nonzero = []
    fold_times = []
    fold_status = []

    for fold_id, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"  Fold {fold_id}/{N_SPLITS} ...", end=" ")

        X_train = X[train_idx]
        X_val = X[val_idx]

        start_time = time.time()

        try:
            model = GraphicalLasso(
                alpha=alpha,
                max_iter=MAX_ITER,
                tol=TOL,
                assume_centered=False,
                mode="cd",
            )
            model.fit(X_train)

            precision_hat = model.precision_
            emp_cov_val = empirical_covariance(X_val)

            val_nll = gaussian_neg_log_likelihood_from_emp_cov(
                emp_cov_val=emp_cov_val,
                precision_hat=precision_hat,
                eps=EPS,
            )

            nnz_offdiag = count_offdiag_nonzero_entries(precision_hat)

            elapsed = time.time() - start_time

            fold_scores.append(val_nll)
            fold_nonzero.append(nnz_offdiag)
            fold_times.append(elapsed)
            fold_status.append("ok")

            print(
                f"ok | val_nll={val_nll:.6f} | "
                f"offdiag_nnz={nnz_offdiag} | time={elapsed:.2f}s"
            )

        except Exception as e:
            elapsed = time.time() - start_time

            fold_scores.append(np.nan)
            fold_nonzero.append(np.nan)
            fold_times.append(elapsed)
            fold_status.append(f"fail: {repr(e)}")

            print(f"failed | time={elapsed:.2f}s | error={repr(e)}")

        finally:
            clear_memory()

    mean_score = float(np.nanmean(fold_scores)) if np.any(~np.isnan(fold_scores)) else np.nan
    std_score = float(np.nanstd(fold_scores)) if np.any(~np.isnan(fold_scores)) else np.nan
    mean_nonzero = float(np.nanmean(fold_nonzero)) if np.any(~np.isnan(fold_nonzero)) else np.nan
    mean_time = float(np.nanmean(fold_times)) if len(fold_times) > 0 else np.nan
    n_success = int(np.sum([s == "ok" for s in fold_status]))
    is_valid_alpha = bool((n_success >= MIN_SUCCESS_FOLDS) and (not np.isnan(mean_score)))

    cv_rows.append({
        "alpha": alpha,
        "mean_val_nll": mean_score,
        "std_val_nll": std_score,
        "mean_offdiag_nnz": mean_nonzero,
        "mean_fit_time_sec": mean_time,
        "n_success_folds": n_success,
        "n_total_folds": N_SPLITS,
        "min_success_folds_required": MIN_SUCCESS_FOLDS,
        "is_valid_alpha": is_valid_alpha,
        "fold_scores": fold_scores,
        "fold_offdiag_nnz": fold_nonzero,
        "fold_fit_time_sec": fold_times,
        "fold_status": fold_status,
    })


# ============================================================
# BUILD RESULTS TABLE
# ============================================================

cv_results_df = pd.DataFrame(cv_rows)

valid_mask = (
    cv_results_df["is_valid_alpha"] == True
) & (
    cv_results_df["mean_val_nll"].notna()
)

if not valid_mask.any():
    raise RuntimeError(
        "No alpha candidate achieved enough successful CV folds. "
        "Try increasing alpha further and/or simplifying the feature set."
    )

cv_results_valid_df = cv_results_df.loc[valid_mask].copy()
cv_results_valid_df.sort_values(
    by=["mean_val_nll", "alpha"],
    ascending=[True, True],
    inplace=True,
)

best_row = cv_results_valid_df.iloc[0]
best_alpha = float(best_row["alpha"])

print("\nCross-validation completed.")
print("Best alpha selected:")
print(f"  alpha             = {best_alpha:.8f}")
print(f"  mean_val_nll      = {best_row['mean_val_nll']:.6f}")
print(f"  std_val_nll       = {best_row['std_val_nll']:.6f}")
print(f"  mean_offdiag_nnz  = {best_row['mean_offdiag_nnz']:.2f}")
print(f"  success_folds     = {int(best_row['n_success_folds'])}/{N_SPLITS}")


# ============================================================
# SAVE RESULTS
# ============================================================

csv_df = cv_results_df.drop(
    columns=["fold_scores", "fold_offdiag_nnz", "fold_fit_time_sec", "fold_status"]
).copy()
csv_df.to_csv(CV_RESULTS_CSV_PATH, index=False)

cv_results_json = []
for row in cv_rows:
    cv_results_json.append({
        "alpha": float(row["alpha"]),
        "mean_val_nll": None if pd.isna(row["mean_val_nll"]) else float(row["mean_val_nll"]),
        "std_val_nll": None if pd.isna(row["std_val_nll"]) else float(row["std_val_nll"]),
        "mean_offdiag_nnz": None if pd.isna(row["mean_offdiag_nnz"]) else float(row["mean_offdiag_nnz"]),
        "mean_fit_time_sec": None if pd.isna(row["mean_fit_time_sec"]) else float(row["mean_fit_time_sec"]),
        "n_success_folds": int(row["n_success_folds"]),
        "n_total_folds": int(row["n_total_folds"]),
        "min_success_folds_required": int(row["min_success_folds_required"]),
        "is_valid_alpha": bool(row["is_valid_alpha"]),
        "fold_scores": [None if pd.isna(x) else float(x) for x in row["fold_scores"]],
        "fold_offdiag_nnz": [None if pd.isna(x) else float(x) for x in row["fold_offdiag_nnz"]],
        "fold_fit_time_sec": [float(x) for x in row["fold_fit_time_sec"]],
        "fold_status": row["fold_status"],
    })

with open(CV_RESULTS_JSON_PATH, "w") as f:
    json.dump(cv_results_json, f, indent=2)

best_alpha_payload = {
    "best_alpha": best_alpha,
    "selection_metric": "mean validation Gaussian negative log-likelihood",
    "min_success_folds_required": int(MIN_SUCCESS_FOLDS),
    "best_result": {
        "alpha": best_alpha,
        "mean_val_nll": float(best_row["mean_val_nll"]),
        "std_val_nll": float(best_row["std_val_nll"]),
        "mean_offdiag_nnz": float(best_row["mean_offdiag_nnz"]),
        "mean_fit_time_sec": float(best_row["mean_fit_time_sec"]),
        "n_success_folds": int(best_row["n_success_folds"]),
        "n_total_folds": int(best_row["n_total_folds"]),
        "is_valid_alpha": bool(best_row["is_valid_alpha"]),
    }
}

with open(BEST_ALPHA_JSON_PATH, "w") as f:
    json.dump(best_alpha_payload, f, indent=2)

print(f"\nSaved CV results CSV        -> {os.path.abspath(CV_RESULTS_CSV_PATH)}")
print(f"Saved CV results JSON       -> {os.path.abspath(CV_RESULTS_JSON_PATH)}")
print(f"Saved best alpha JSON       -> {os.path.abspath(BEST_ALPHA_JSON_PATH)}")


# ============================================================
# METADATA
# ============================================================

meta = {
    "module": "MODULE 3 - STATE GRAPH - GRAPHICAL LASSO CV FOR ALPHA SELECTION",
    "input_x_lagged_std_npy": X_LAGGED_STD_NPY_PATH,
    "input_lagged_feature_names_json": LAGGED_FEATURES_JSON_PATH,
    "n_samples": int(n_samples),
    "n_features": int(n_features),
    "alpha_grid": [float(a) for a in ALPHA_GRID],
    "n_splits": int(N_SPLITS),
    "min_success_folds": int(MIN_SUCCESS_FOLDS),
    "shuffle": bool(SHUFFLE),
    "random_state": None if not SHUFFLE else int(RANDOM_STATE),
    "max_iter": int(MAX_ITER),
    "tol": float(TOL),
    "eps": float(EPS),
    "use_float32": bool(USE_FLOAT32),
    "cv_results_csv": CV_RESULTS_CSV_PATH,
    "cv_results_json": CV_RESULTS_JSON_PATH,
    "best_alpha_json": BEST_ALPHA_JSON_PATH,
}

with open(META_JSON_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata              -> {os.path.abspath(META_JSON_PATH)}")


# ============================================================
# QUICK CHECKS
# ============================================================

best_rank = int(cv_results_valid_df.reset_index(drop=True).index[
    cv_results_valid_df.reset_index(drop=True)["alpha"] == best_alpha
][0]) + 1

print("\nQuick checks:")
print(f"Number of tested alphas      : {len(ALPHA_GRID)}")
print(f"Number of valid alphas       : {len(cv_results_valid_df)}")
print(f"Required success folds       : {MIN_SUCCESS_FOLDS}/{N_SPLITS}")
print(f"Selected alpha rank          : {best_rank}")
print(f"Best alpha                   : {best_alpha:.8f}")
print(f"Best mean validation NLL     : {best_row['mean_val_nll']:.6f}")

print("\nDONE: Graphical Lasso CV completed successfully.")

In [ ]:
# ============================================================
# MODULE 4
# STATE GRAPH - FIT FINAL GRAPHICAL LASSO AND EXPORT GRAPH
# input : pruned lagged standardized cache + best alpha from MODULE 3
# output: covariance, precision, partial correlation, adjacency, edge list
# ============================================================

import os
import gc
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.covariance import GraphicalLasso


# ============================================================
# CONFIG
# ============================================================

GLASSO_CV_DIR = "./glasso_cache_v4/state_graph_glasso_cv"
LAGGED_CACHE_DIR = "./glasso_cache_v4/lagged_cache_pruned"

X_LAGGED_STD_NPY_PATH = os.path.join(LAGGED_CACHE_DIR, "lob_state_lagged_Xstd_pruned.npy")
LAGGED_FEATURES_JSON_PATH = os.path.join(LAGGED_CACHE_DIR, "lob_state_lagged_feature_names_pruned.json")
BEST_ALPHA_JSON_PATH = os.path.join(GLASSO_CV_DIR, "lob_state_glasso_best_alpha.json")

MAX_ITER = 400
TOL = 1e-4
EPS = 1e-12
EDGE_THRESHOLD = 1e-12
TOP_K_EDGES = 500

USE_FLOAT32 = True

OUTDIR = "./glasso_cache_v4/state_graph_final"
os.makedirs(OUTDIR, exist_ok=True)

COVARIANCE_NPY_PATH = os.path.join(OUTDIR, "lob_state_glasso_covariance.npy")
PRECISION_NPY_PATH = os.path.join(OUTDIR, "lob_state_glasso_precision.npy")
PARTIAL_CORR_NPY_PATH = os.path.join(OUTDIR, "lob_state_glasso_partial_correlation.npy")
ADJACENCY_NPY_PATH = os.path.join(OUTDIR, "lob_state_glasso_adjacency.npy")

COVARIANCE_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_covariance.csv")
PRECISION_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_precision.csv")
PARTIAL_CORR_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_partial_correlation.csv")
ADJACENCY_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_adjacency.csv")

EDGE_LIST_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_edge_list.csv")
TOP_EDGES_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_top_edges.csv")

SUMMARY_JSON_PATH = os.path.join(OUTDIR, "lob_state_glasso_summary.json")
META_JSON_PATH = os.path.join(OUTDIR, "lob_state_glasso_metadata.json")


# ============================================================
# UTILS
# ============================================================

def clear_memory():
    """
    Trigger Python garbage collection.
    """
    gc.collect()


def safe_logdet(A: np.ndarray, eps: float = 1e-12) -> float:
    """
    Compute a numerically safe log-determinant.

    Parameters
    ----------
    A : np.ndarray
        Symmetric positive definite matrix.
    eps : float
        Small diagonal jitter if needed.

    Returns
    -------
    float
        Log-determinant of A.
    """
    A64 = A.astype(np.float64, copy=False)
    sign, logdet = np.linalg.slogdet(A64)

    if sign > 0:
        return float(logdet)

    A_jitter = A64 + eps * np.eye(A64.shape[0], dtype=np.float64)
    sign_j, logdet_j = np.linalg.slogdet(A_jitter)

    if sign_j <= 0:
        raise ValueError("Matrix is not positive definite even after jitter.")

    return float(logdet_j)


def precision_to_partial_correlation(
    precision: np.ndarray,
    eps: float = 1e-12
) -> np.ndarray:
    """
    Convert a precision matrix to a partial correlation matrix.

    Formula:
        rho_ij = -Theta_ij / sqrt(Theta_ii * Theta_jj), for i != j
        rho_ii = 1

    Parameters
    ----------
    precision : np.ndarray
        Precision matrix.
    eps : float
        Small constant for numerical stability.

    Returns
    -------
    np.ndarray
        Partial correlation matrix.
    """
    Theta = precision.astype(np.float64, copy=False)
    diag = np.clip(np.diag(Theta), eps, None)
    denom = np.sqrt(np.outer(diag, diag))

    partial_corr = -Theta / denom
    np.fill_diagonal(partial_corr, 1.0)

    partial_corr = np.clip(partial_corr, -1.0, 1.0)
    return partial_corr


def precision_to_adjacency(
    precision: np.ndarray,
    threshold: float = 1e-12
) -> np.ndarray:
    """
    Build a binary adjacency matrix from the precision matrix.

    An undirected edge exists when the absolute off-diagonal precision entry
    is strictly larger than the threshold.
    """
    A = (np.abs(precision) > threshold).astype(np.int8)
    np.fill_diagonal(A, 0)
    A = np.maximum(A, A.T).astype(np.int8)
    return A


def build_edge_list(
    precision: np.ndarray,
    partial_corr: np.ndarray,
    feature_names: list[str],
    threshold: float = 1e-12
) -> pd.DataFrame:
    """
    Build the undirected edge list from precision and partial correlation matrices.

    Only upper-triangular pairs (i < j) are included.
    """
    n = precision.shape[0]
    rows = []

    for i in range(n):
        for j in range(i + 1, n):
            w_precision = float(precision[i, j])

            if abs(w_precision) <= threshold:
                continue

            w_partial = float(partial_corr[i, j])

            rows.append({
                "source": feature_names[i],
                "target": feature_names[j],
                "precision_weight": w_precision,
                "abs_precision_weight": abs(w_precision),
                "partial_corr_weight": w_partial,
                "abs_partial_corr_weight": abs(w_partial),
                "sign": "positive" if w_partial > 0 else "negative",
            })

    edge_df = pd.DataFrame(rows)

    if not edge_df.empty:
        edge_df.sort_values(
            by=["abs_precision_weight", "source", "target"],
            ascending=[False, True, True],
            inplace=True,
        )
        edge_df.reset_index(drop=True, inplace=True)

    return edge_df


def matrix_symmetry_error(M: np.ndarray) -> float:
    """
    Compute the maximum absolute symmetry error of a matrix.
    """
    return float(np.max(np.abs(M - M.T)))


def count_offdiag_nonzero_entries(
    M: np.ndarray,
    threshold: float = 1e-12
) -> int:
    """
    Count non-zero off-diagonal entries in a square matrix.
    """
    X = M.copy()
    np.fill_diagonal(X, 0.0)
    return int(np.sum(np.abs(X) > threshold))


# ============================================================
# LOAD INPUTS
# ============================================================

print("Loading pruned lagged standardized matrix and best alpha...")

if not os.path.exists(X_LAGGED_STD_NPY_PATH):
    raise FileNotFoundError(f"Missing pruned lagged standardized matrix: {X_LAGGED_STD_NPY_PATH}")

if not os.path.exists(LAGGED_FEATURES_JSON_PATH):
    raise FileNotFoundError(f"Missing pruned lagged feature names: {LAGGED_FEATURES_JSON_PATH}")

if not os.path.exists(BEST_ALPHA_JSON_PATH):
    raise FileNotFoundError(f"Missing best alpha file: {BEST_ALPHA_JSON_PATH}")

X = np.load(X_LAGGED_STD_NPY_PATH)

with open(LAGGED_FEATURES_JSON_PATH, "r") as f:
    feature_names = json.load(f)

with open(BEST_ALPHA_JSON_PATH, "r") as f:
    best_alpha_payload = json.load(f)

best_alpha = float(best_alpha_payload["best_alpha"])

print(f"Loaded X shape              : {X.shape}")
print(f"Loaded number of features   : {len(feature_names)}")
print(f"Loaded best alpha           : {best_alpha:.8f}")


# ============================================================
# BASIC VALIDATION
# ============================================================

if X.ndim != 2:
    raise ValueError(f"X must be 2D, got shape {X.shape}")

n_samples, n_features = X.shape

if n_features != len(feature_names):
    raise ValueError(
        "Mismatch between X columns and feature names.\n"
        f"X columns            : {n_features}\n"
        f"Feature names count  : {len(feature_names)}"
    )

if best_alpha <= 0:
    raise ValueError(f"best_alpha must be > 0, got {best_alpha}")

dtype = np.float32 if USE_FLOAT32 else np.float64
X = X.astype(dtype, copy=False)

print("\nInput validation passed.")
print(f"n_samples      = {n_samples}")
print(f"n_features     = {n_features}")
print(f"MAX_ITER       = {MAX_ITER}")
print(f"TOL            = {TOL}")
print(f"EDGE_THRESHOLD = {EDGE_THRESHOLD}")


# ============================================================
# FIT FINAL GRAPHICAL LASSO
# ============================================================

print("\nFitting final Graphical Lasso model...")

start_time = time.time()

model = GraphicalLasso(
    alpha=best_alpha,
    max_iter=MAX_ITER,
    tol=TOL,
    assume_centered=False,
    mode="cd",
)
model.fit(X)

fit_time_sec = time.time() - start_time

covariance_hat = model.covariance_.astype(np.float64, copy=False)
precision_hat = model.precision_.astype(np.float64, copy=False)

print(f"Model fit completed in {fit_time_sec:.2f} seconds.")


# ============================================================
# DERIVED MATRICES
# ============================================================

print("\nBuilding derived graph matrices...")

partial_corr_hat = precision_to_partial_correlation(precision_hat, eps=EPS)
adjacency_hat = precision_to_adjacency(precision_hat, threshold=EDGE_THRESHOLD)

print("Derived matrices created successfully.")


# ============================================================
# SAVE MATRICES
# ============================================================

print("\nSaving covariance / precision / partial correlation / adjacency...")

np.save(COVARIANCE_NPY_PATH, covariance_hat)
np.save(PRECISION_NPY_PATH, precision_hat)
np.save(PARTIAL_CORR_NPY_PATH, partial_corr_hat)
np.save(ADJACENCY_NPY_PATH, adjacency_hat)

cov_df = pd.DataFrame(covariance_hat, index=feature_names, columns=feature_names)
prec_df = pd.DataFrame(precision_hat, index=feature_names, columns=feature_names)
pcorr_df = pd.DataFrame(partial_corr_hat, index=feature_names, columns=feature_names)
adj_df = pd.DataFrame(adjacency_hat, index=feature_names, columns=feature_names)

cov_df.to_csv(COVARIANCE_CSV_PATH, index=True)
prec_df.to_csv(PRECISION_CSV_PATH, index=True)
pcorr_df.to_csv(PARTIAL_CORR_CSV_PATH, index=True)
adj_df.to_csv(ADJACENCY_CSV_PATH, index=True)

print(f"Saved covariance NPY        -> {os.path.abspath(COVARIANCE_NPY_PATH)}")
print(f"Saved precision NPY         -> {os.path.abspath(PRECISION_NPY_PATH)}")
print(f"Saved partial corr NPY      -> {os.path.abspath(PARTIAL_CORR_NPY_PATH)}")
print(f"Saved adjacency NPY         -> {os.path.abspath(ADJACENCY_NPY_PATH)}")
print(f"Saved covariance CSV        -> {os.path.abspath(COVARIANCE_CSV_PATH)}")
print(f"Saved precision CSV         -> {os.path.abspath(PRECISION_CSV_PATH)}")
print(f"Saved partial corr CSV      -> {os.path.abspath(PARTIAL_CORR_CSV_PATH)}")
print(f"Saved adjacency CSV         -> {os.path.abspath(ADJACENCY_CSV_PATH)}")


# ============================================================
# BUILD EDGE LIST
# ============================================================

print("\nBuilding edge list...")

edge_df = build_edge_list(
    precision=precision_hat,
    partial_corr=partial_corr_hat,
    feature_names=feature_names,
    threshold=EDGE_THRESHOLD,
)

edge_df.to_csv(EDGE_LIST_CSV_PATH, index=False)

if edge_df.empty:
    top_edges_df = edge_df.copy()
else:
    top_edges_df = edge_df.head(TOP_K_EDGES).copy()

top_edges_df.to_csv(TOP_EDGES_CSV_PATH, index=False)

n_edges = int(len(edge_df))

print(f"Number of edges             : {n_edges}")
print(f"Saved edge list             -> {os.path.abspath(EDGE_LIST_CSV_PATH)}")
print(f"Saved top edges             -> {os.path.abspath(TOP_EDGES_CSV_PATH)}")


# ============================================================
# SUMMARY
# ============================================================

n_possible_edges = n_features * (n_features - 1) // 2
graph_density = float(n_edges / n_possible_edges) if n_possible_edges > 0 else 0.0

precision_offdiag_nnz = count_offdiag_nonzero_entries(
    precision_hat,
    threshold=EDGE_THRESHOLD
)
adjacency_undirected_edges = int(np.sum(adjacency_hat) // 2)

diag_mean_cov = float(np.mean(np.diag(covariance_hat)))
diag_mean_prec = float(np.mean(np.diag(precision_hat)))
diag_mean_pcorr = float(np.mean(np.diag(partial_corr_hat)))

symmetry_error_cov = matrix_symmetry_error(covariance_hat)
symmetry_error_prec = matrix_symmetry_error(precision_hat)
symmetry_error_pcorr = matrix_symmetry_error(partial_corr_hat)
symmetry_error_adj = matrix_symmetry_error(adjacency_hat.astype(np.float64))

summary = {
    "best_alpha": best_alpha,
    "fit_time_sec": float(fit_time_sec),
    "n_samples": int(n_samples),
    "n_features": int(n_features),
    "n_possible_edges": int(n_possible_edges),
    "n_edges": int(n_edges),
    "graph_density": float(graph_density),
    "precision_offdiag_nnz": int(precision_offdiag_nnz),
    "adjacency_undirected_edges": int(adjacency_undirected_edges),
    "diag_mean_covariance": float(diag_mean_cov),
    "diag_mean_precision": float(diag_mean_prec),
    "diag_mean_partial_corr": float(diag_mean_pcorr),
    "symmetry_error_covariance": float(symmetry_error_cov),
    "symmetry_error_precision": float(symmetry_error_prec),
    "symmetry_error_partial_corr": float(symmetry_error_pcorr),
    "symmetry_error_adjacency": float(symmetry_error_adj),
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved summary               -> {os.path.abspath(SUMMARY_JSON_PATH)}")


# ============================================================
# METADATA
# ============================================================

meta = {
    "module": "MODULE 4 - STATE GRAPH - FIT FINAL GRAPHICAL LASSO AND EXPORT GRAPH",
    "input_x_lagged_std_npy": X_LAGGED_STD_NPY_PATH,
    "input_feature_names_json": LAGGED_FEATURES_JSON_PATH,
    "input_best_alpha_json": BEST_ALPHA_JSON_PATH,
    "best_alpha": float(best_alpha),
    "max_iter": int(MAX_ITER),
    "tol": float(TOL),
    "eps": float(EPS),
    "edge_threshold": float(EDGE_THRESHOLD),
    "top_k_edges": int(TOP_K_EDGES),
    "use_float32": bool(USE_FLOAT32),
    "covariance_npy": COVARIANCE_NPY_PATH,
    "precision_npy": PRECISION_NPY_PATH,
    "partial_corr_npy": PARTIAL_CORR_NPY_PATH,
    "adjacency_npy": ADJACENCY_NPY_PATH,
    "covariance_csv": COVARIANCE_CSV_PATH,
    "precision_csv": PRECISION_CSV_PATH,
    "partial_corr_csv": PARTIAL_CORR_CSV_PATH,
    "adjacency_csv": ADJACENCY_CSV_PATH,
    "edge_list_csv": EDGE_LIST_CSV_PATH,
    "top_edges_csv": TOP_EDGES_CSV_PATH,
    "summary_json": SUMMARY_JSON_PATH,
}

with open(META_JSON_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata              -> {os.path.abspath(META_JSON_PATH)}")


# ============================================================
# QUICK CHECKS
# ============================================================

print("\nQuick checks:")
print(f"Best alpha                    : {best_alpha:.8f}")
print(f"Fit time (sec)                : {fit_time_sec:.2f}")
print(f"Number of undirected edges    : {n_edges}")
print(f"Graph density                 : {graph_density:.8f}")
print(f"Adjacency edges (check)       : {adjacency_undirected_edges}")
print(f"Precision offdiag nnz         : {precision_offdiag_nnz}")
print(f"Covariance symmetry error     : {symmetry_error_cov:.6e}")
print(f"Precision symmetry error      : {symmetry_error_prec:.6e}")
print(f"Partial corr symmetry error   : {symmetry_error_pcorr:.6e}")
print(f"Adjacency symmetry error      : {symmetry_error_adj:.6e}")

if n_edges != adjacency_undirected_edges:
    raise ValueError(
        f"Edge count mismatch: edge_df={n_edges}, adjacency={adjacency_undirected_edges}"
    )

print("\nDONE: final Glasso graph exported successfully.")

In [ ]:
# ============================================================
# MODULE 5A
# STATE GRAPH - STRUCTURAL EDGE ANALYSIS
# input : final edge list from MODULE 4
# output: parsed edge table + structural summaries by category
# ============================================================

import os
import gc
import json
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

FINAL_GRAPH_DIR = "./glasso_cache_v4/state_graph_final"

EDGE_LIST_CSV_PATH = os.path.join(FINAL_GRAPH_DIR, "lob_state_glasso_edge_list.csv")

TOP_K_FOR_PREVIEW = 200

OUTDIR = "./glasso_cache_v4/state_graph_edge_analysis"
os.makedirs(OUTDIR, exist_ok=True)

PARSED_EDGE_LIST_CSV_PATH = os.path.join(OUTDIR, "lob_state_glasso_edge_list_parsed.csv")

SUMMARY_BY_LAG_TYPE_CSV_PATH = os.path.join(OUTDIR, "summary_by_lag_type.csv")
SUMMARY_BY_LEVEL_REL_CSV_PATH = os.path.join(OUTDIR, "summary_by_level_relation.csv")
SUMMARY_BY_FEATURE_PAIR_CSV_PATH = os.path.join(OUTDIR, "summary_by_feature_pair.csv")
SUMMARY_BY_SIDE_PAIR_CSV_PATH = os.path.join(OUTDIR, "summary_by_side_pair.csv")
SUMMARY_BY_TOUCHES_LAG0_CSV_PATH = os.path.join(OUTDIR, "summary_by_touches_lag0.csv")

TOP_EDGES_BY_ABS_PRECISION_CSV_PATH = os.path.join(OUTDIR, "top_edges_by_abs_precision_with_categories.csv")
GLOBAL_SUMMARY_JSON_PATH = os.path.join(OUTDIR, "lob_state_edge_analysis_summary.json")
META_JSON_PATH = os.path.join(OUTDIR, "lob_state_edge_analysis_metadata.json")


# ============================================================
# UTILS
# ============================================================

def clear_memory():
    """
    Trigger Python garbage collection.
    """
    gc.collect()


def parse_lagged_feature_name(feature_name: str) -> dict:
    """
    Parse lagged feature names.

    Supported formats:

    1) Per-level raw features:
        askp_0_lag_0
        asks_3_lag_2
        bidp_7_lag_1
        bids_9_lag_0

    2) Global semantic features:
        mid_price_lag_0
        spread_lag_1
        depth_ask_total_lag_2
        depth_bid_total_lag_3
        global_imbalance_lag_0

    Returns
    -------
    dict
        Parsed components:
        - original_name
        - feature_type
        - base_feature
        - feature_family
        - side_family
        - level
        - lag
    """

    # --------------------------------------------------------
    # Case 1: per-level raw features
    # --------------------------------------------------------
    pattern_level = r"^(askp|asks|bidp|bids)_(\d+)_lag_(\d+)$"
    match_level = re.match(pattern_level, feature_name)

    if match_level is not None:
        base_feature = match_level.group(1)
        level = int(match_level.group(2))
        lag = int(match_level.group(3))

        if base_feature == "askp":
            feature_family = "price"
            side_family = "ask"
        elif base_feature == "asks":
            feature_family = "size"
            side_family = "ask"
        elif base_feature == "bidp":
            feature_family = "price"
            side_family = "bid"
        elif base_feature == "bids":
            feature_family = "size"
            side_family = "bid"
        else:
            raise ValueError(f"Unknown base feature: {base_feature}")

        return {
            "original_name": feature_name,
            "feature_type": "level",
            "base_feature": base_feature,
            "feature_family": feature_family,
            "side_family": side_family,
            "level": level,
            "lag": lag,
        }

    # --------------------------------------------------------
    # Case 2: global semantic features
    # --------------------------------------------------------
    pattern_global = r"^(mid_price|spread|depth_ask_total|depth_bid_total|global_imbalance)_lag_(\d+)$"
    match_global = re.match(pattern_global, feature_name)

    if match_global is not None:
        base_feature = match_global.group(1)
        lag = int(match_global.group(2))

        if base_feature == "mid_price":
            feature_family = "price_global"
            side_family = "global"
        elif base_feature == "spread":
            feature_family = "spread"
            side_family = "global"
        elif base_feature == "depth_ask_total":
            feature_family = "depth"
            side_family = "ask"
        elif base_feature == "depth_bid_total":
            feature_family = "depth"
            side_family = "bid"
        elif base_feature == "global_imbalance":
            feature_family = "imbalance"
            side_family = "global"
        else:
            raise ValueError(f"Unknown global base feature: {base_feature}")

        return {
            "original_name": feature_name,
            "feature_type": "global",
            "base_feature": base_feature,
            "feature_family": feature_family,
            "side_family": side_family,
            "level": -1,   # no explicit level
            "lag": lag,
        }

    raise ValueError(f"Unsupported feature name format: {feature_name}")


def canonical_pair(a: str, b: str) -> str:
    """
    Return a canonical pair label sorted alphabetically.
    """
    x, y = sorted([str(a), str(b)])
    return f"{x}|{y}"


def classify_level_relation(level_diff: int, src_type: str, tgt_type: str) -> str:
    """
    Classify the relation between two levels.

    If at least one feature is global (no explicit level), return 'global_level'.
    """
    if src_type == "global" or tgt_type == "global":
        return "global_level"

    if level_diff == 0:
        return "same_level"
    if level_diff == 1:
        return "adjacent_level"
    return "distant_level"


def classify_lag_relation(lag_diff: int) -> str:
    """
    Classify the relation between two lags.
    """
    if lag_diff == 0:
        return "intra_lag"
    if lag_diff == 1:
        return "cross_lag_1"
    return "cross_lag_gt1"


def aggregate_edge_summary(
    df: pd.DataFrame,
    group_col: str
) -> pd.DataFrame:
    """
    Aggregate structural edge statistics by one categorical column.
    """
    if df.empty:
        return pd.DataFrame(columns=[
            group_col,
            "n_edges",
            "share_edges",
            "mean_abs_precision_weight",
            "median_abs_precision_weight",
            "max_abs_precision_weight",
            "mean_abs_partial_corr_weight",
            "median_abs_partial_corr_weight",
            "max_abs_partial_corr_weight",
        ])

    out = (
        df.groupby(group_col, dropna=False)
        .agg(
            n_edges=("source", "size"),
            mean_abs_precision_weight=("abs_precision_weight", "mean"),
            median_abs_precision_weight=("abs_precision_weight", "median"),
            max_abs_precision_weight=("abs_precision_weight", "max"),
            mean_abs_partial_corr_weight=("abs_partial_corr_weight", "mean"),
            median_abs_partial_corr_weight=("abs_partial_corr_weight", "median"),
            max_abs_partial_corr_weight=("abs_partial_corr_weight", "max"),
        )
        .reset_index()
    )

    total_edges = len(df)
    out["share_edges"] = out["n_edges"] / total_edges

    out.sort_values(
        by=["n_edges", "mean_abs_precision_weight", group_col],
        ascending=[False, False, True],
        inplace=True,
    )
    out.reset_index(drop=True, inplace=True)

    cols = [
        group_col,
        "n_edges",
        "share_edges",
        "mean_abs_precision_weight",
        "median_abs_precision_weight",
        "max_abs_precision_weight",
        "mean_abs_partial_corr_weight",
        "median_abs_partial_corr_weight",
        "max_abs_partial_corr_weight",
    ]
    return out[cols]


# ============================================================
# LOAD EDGE LIST
# ============================================================

print("Loading final edge list from MODULE 4...")

if not os.path.exists(EDGE_LIST_CSV_PATH):
    raise FileNotFoundError(f"Missing edge list: {EDGE_LIST_CSV_PATH}")

edge_df = pd.read_csv(EDGE_LIST_CSV_PATH)

print(f"Loaded edge list shape      : {edge_df.shape}")


# ============================================================
# BASIC VALIDATION
# ============================================================

required_cols = {
    "source",
    "target",
    "precision_weight",
    "abs_precision_weight",
    "partial_corr_weight",
    "abs_partial_corr_weight",
    "sign",
}

missing_cols = required_cols - set(edge_df.columns)
if missing_cols:
    raise ValueError(f"Missing required edge list columns: {sorted(missing_cols)}")

if edge_df.empty:
    raise ValueError("Edge list is empty. Nothing to analyze.")

print("\nInput validation passed.")
print(f"Number of edges = {len(edge_df)}")


# ============================================================
# PARSE SOURCE / TARGET NODES
# ============================================================

print("\nParsing lagged feature names and building structural categories...")

source_info = edge_df["source"].apply(parse_lagged_feature_name).apply(pd.Series)
target_info = edge_df["target"].apply(parse_lagged_feature_name).apply(pd.Series)

source_info = source_info.add_prefix("src_")
target_info = target_info.add_prefix("tgt_")

parsed_df = pd.concat([edge_df.copy(), source_info, target_info], axis=1)

parsed_df["level_diff"] = (parsed_df["src_level"] - parsed_df["tgt_level"]).abs()
parsed_df["lag_diff"] = (parsed_df["src_lag"] - parsed_df["tgt_lag"]).abs()

parsed_df["level_relation"] = parsed_df.apply(
    lambda row: classify_level_relation(
        row["level_diff"],
        row["src_feature_type"],
        row["tgt_feature_type"],
    ),
    axis=1,
)
parsed_df["lag_relation"] = parsed_df["lag_diff"].apply(classify_lag_relation)

parsed_df["same_base_feature"] = parsed_df["src_base_feature"] == parsed_df["tgt_base_feature"]
parsed_df["same_feature_family"] = parsed_df["src_feature_family"] == parsed_df["tgt_feature_family"]
parsed_df["same_side_family"] = parsed_df["src_side_family"] == parsed_df["tgt_side_family"]
parsed_df["same_feature_type"] = parsed_df["src_feature_type"] == parsed_df["tgt_feature_type"]

parsed_df["touches_lag0"] = (parsed_df["src_lag"] == 0) | (parsed_df["tgt_lag"] == 0)
parsed_df["both_lag0"] = (parsed_df["src_lag"] == 0) & (parsed_df["tgt_lag"] == 0)
parsed_df["crosses_time"] = parsed_df["lag_diff"] > 0

parsed_df["feature_pair"] = parsed_df.apply(
    lambda row: canonical_pair(row["src_base_feature"], row["tgt_base_feature"]),
    axis=1
)
parsed_df["feature_family_pair"] = parsed_df.apply(
    lambda row: canonical_pair(row["src_feature_family"], row["tgt_feature_family"]),
    axis=1
)
parsed_df["side_pair"] = parsed_df.apply(
    lambda row: canonical_pair(row["src_side_family"], row["tgt_side_family"]),
    axis=1
)
parsed_df["feature_type_pair"] = parsed_df.apply(
    lambda row: canonical_pair(row["src_feature_type"], row["tgt_feature_type"]),
    axis=1
)

parsed_df["same_level_flag"] = (
    (parsed_df["level_diff"] == 0)
    & (parsed_df["src_feature_type"] == "level")
    & (parsed_df["tgt_feature_type"] == "level")
)
parsed_df["adjacent_level_flag"] = (
    (parsed_df["level_diff"] == 1)
    & (parsed_df["src_feature_type"] == "level")
    & (parsed_df["tgt_feature_type"] == "level")
)
parsed_df["same_lag_flag"] = parsed_df["lag_diff"] == 0

print("Parsing completed successfully.")


# ============================================================
# SAVE PARSED EDGE LIST
# ============================================================

parsed_df.to_csv(PARSED_EDGE_LIST_CSV_PATH, index=False)
print(f"\nSaved parsed edge list      -> {os.path.abspath(PARSED_EDGE_LIST_CSV_PATH)}")


# ============================================================
# STRUCTURAL SUMMARIES
# ============================================================

print("\nComputing structural summaries...")

summary_by_lag_type = aggregate_edge_summary(parsed_df, "lag_relation")
summary_by_level_rel = aggregate_edge_summary(parsed_df, "level_relation")
summary_by_feature_pair = aggregate_edge_summary(parsed_df, "feature_pair")
summary_by_side_pair = aggregate_edge_summary(parsed_df, "side_pair")
summary_by_touches_lag0 = aggregate_edge_summary(parsed_df, "touches_lag0")

summary_by_lag_type.to_csv(SUMMARY_BY_LAG_TYPE_CSV_PATH, index=False)
summary_by_level_rel.to_csv(SUMMARY_BY_LEVEL_REL_CSV_PATH, index=False)
summary_by_feature_pair.to_csv(SUMMARY_BY_FEATURE_PAIR_CSV_PATH, index=False)
summary_by_side_pair.to_csv(SUMMARY_BY_SIDE_PAIR_CSV_PATH, index=False)
summary_by_touches_lag0.to_csv(SUMMARY_BY_TOUCHES_LAG0_CSV_PATH, index=False)

print(f"Saved summary by lag type   -> {os.path.abspath(SUMMARY_BY_LAG_TYPE_CSV_PATH)}")
print(f"Saved summary by level rel  -> {os.path.abspath(SUMMARY_BY_LEVEL_REL_CSV_PATH)}")
print(f"Saved summary by feat pair  -> {os.path.abspath(SUMMARY_BY_FEATURE_PAIR_CSV_PATH)}")
print(f"Saved summary by side pair  -> {os.path.abspath(SUMMARY_BY_SIDE_PAIR_CSV_PATH)}")
print(f"Saved summary by lag0 touch -> {os.path.abspath(SUMMARY_BY_TOUCHES_LAG0_CSV_PATH)}")


# ============================================================
# TOP-K PREVIEW TABLE
# ============================================================

top_preview_df = parsed_df.sort_values(
    by=["abs_precision_weight", "source", "target"],
    ascending=[False, True, True],
).head(TOP_K_FOR_PREVIEW).copy()

top_preview_df.to_csv(TOP_EDGES_BY_ABS_PRECISION_CSV_PATH, index=False)

print(f"Saved top-k parsed edges    -> {os.path.abspath(TOP_EDGES_BY_ABS_PRECISION_CSV_PATH)}")


# ============================================================
# GLOBAL SUMMARY
# ============================================================

n_edges = int(len(parsed_df))

n_intra_lag = int((parsed_df["lag_relation"] == "intra_lag").sum())
n_cross_lag = int((parsed_df["lag_relation"] != "intra_lag").sum())

n_same_level = int((parsed_df["level_relation"] == "same_level").sum())
n_adjacent_level = int((parsed_df["level_relation"] == "adjacent_level").sum())
n_distant_level = int((parsed_df["level_relation"] == "distant_level").sum())
n_global_level = int((parsed_df["level_relation"] == "global_level").sum())

n_touches_lag0 = int(parsed_df["touches_lag0"].sum())
n_both_lag0 = int(parsed_df["both_lag0"].sum())
n_crosses_time = int(parsed_df["crosses_time"].sum())

global_summary = {
    "n_edges": n_edges,
    "n_intra_lag": n_intra_lag,
    "share_intra_lag": float(n_intra_lag / n_edges),
    "n_cross_lag": n_cross_lag,
    "share_cross_lag": float(n_cross_lag / n_edges),
    "n_same_level": n_same_level,
    "share_same_level": float(n_same_level / n_edges),
    "n_adjacent_level": n_adjacent_level,
    "share_adjacent_level": float(n_adjacent_level / n_edges),
    "n_distant_level": n_distant_level,
    "share_distant_level": float(n_distant_level / n_edges),
    "n_global_level": n_global_level,
    "share_global_level": float(n_global_level / n_edges),
    "n_touches_lag0": n_touches_lag0,
    "share_touches_lag0": float(n_touches_lag0 / n_edges),
    "n_both_lag0": n_both_lag0,
    "share_both_lag0": float(n_both_lag0 / n_edges),
    "n_crosses_time": n_crosses_time,
    "share_crosses_time": float(n_crosses_time / n_edges),
    "mean_abs_precision_weight": float(parsed_df["abs_precision_weight"].mean()),
    "median_abs_precision_weight": float(parsed_df["abs_precision_weight"].median()),
    "max_abs_precision_weight": float(parsed_df["abs_precision_weight"].max()),
    "mean_abs_partial_corr_weight": float(parsed_df["abs_partial_corr_weight"].mean()),
    "median_abs_partial_corr_weight": float(parsed_df["abs_partial_corr_weight"].median()),
    "max_abs_partial_corr_weight": float(parsed_df["abs_partial_corr_weight"].max()),
}

with open(GLOBAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(global_summary, f, indent=2)

print(f"\nSaved global summary        -> {os.path.abspath(GLOBAL_SUMMARY_JSON_PATH)}")


# ============================================================
# METADATA
# ============================================================

meta = {
    "module": "MODULE 5A - STATE GRAPH - STRUCTURAL EDGE ANALYSIS",
    "input_edge_list_csv": EDGE_LIST_CSV_PATH,
    "top_k_for_preview": int(TOP_K_FOR_PREVIEW),
    "parsed_edge_list_csv": PARSED_EDGE_LIST_CSV_PATH,
    "summary_by_lag_type_csv": SUMMARY_BY_LAG_TYPE_CSV_PATH,
    "summary_by_level_relation_csv": SUMMARY_BY_LEVEL_REL_CSV_PATH,
    "summary_by_feature_pair_csv": SUMMARY_BY_FEATURE_PAIR_CSV_PATH,
    "summary_by_side_pair_csv": SUMMARY_BY_SIDE_PAIR_CSV_PATH,
    "summary_by_touches_lag0_csv": SUMMARY_BY_TOUCHES_LAG0_CSV_PATH,
    "top_edges_by_abs_precision_csv": TOP_EDGES_BY_ABS_PRECISION_CSV_PATH,
    "global_summary_json": GLOBAL_SUMMARY_JSON_PATH,
}

with open(META_JSON_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata              -> {os.path.abspath(META_JSON_PATH)}")


# ============================================================
# QUICK CHECKS
# ============================================================

print("\nQuick checks:")
print(f"Total edges                  : {n_edges}")
print(f"Intra-lag edges              : {n_intra_lag} ({n_intra_lag / n_edges:.4%})")
print(f"Cross-lag edges              : {n_cross_lag} ({n_cross_lag / n_edges:.4%})")
print(f"Same-level edges             : {n_same_level} ({n_same_level / n_edges:.4%})")
print(f"Adjacent-level edges         : {n_adjacent_level} ({n_adjacent_level / n_edges:.4%})")
print(f"Distant-level edges          : {n_distant_level} ({n_distant_level / n_edges:.4%})")
print(f"Global-level edges           : {n_global_level} ({n_global_level / n_edges:.4%})")
print(f"Edges touching lag 0         : {n_touches_lag0} ({n_touches_lag0 / n_edges:.4%})")
print(f"Edges fully inside lag 0     : {n_both_lag0} ({n_both_lag0 / n_edges:.4%})")
print(f"Edges crossing time          : {n_crosses_time} ({n_crosses_time / n_edges:.4%})")

print("\nDONE: structural edge analysis completed successfully.")

In [ ]:
# ============================================================
# MODULE 5B
# STATE GRAPH - INTERPRETABLE EDGE RANKINGS
# input : parsed edge list from MODULE 5A
# output: top edge rankings by structural category
# ============================================================

import os
import gc
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

EDGE_ANALYSIS_DIR = "./glasso_cache_v4/state_graph_edge_analysis"

PARSED_EDGE_LIST_CSV_PATH = os.path.join(
    EDGE_ANALYSIS_DIR,
    "lob_state_glasso_edge_list_parsed.csv"
)

TOP_K = 100

OUTDIR = "./glasso_cache_v4/state_graph_edge_rankings"
os.makedirs(OUTDIR, exist_ok=True)

TOP_OVERALL_CSV_PATH = os.path.join(OUTDIR, "top_edges_overall.csv")
TOP_INTRA_LAG_CSV_PATH = os.path.join(OUTDIR, "top_edges_intra_lag.csv")
TOP_CROSS_LAG_CSV_PATH = os.path.join(OUTDIR, "top_edges_cross_lag.csv")

TOP_SAME_LEVEL_CSV_PATH = os.path.join(OUTDIR, "top_edges_same_level.csv")
TOP_ADJACENT_LEVEL_CSV_PATH = os.path.join(OUTDIR, "top_edges_adjacent_level.csv")
TOP_DISTANT_LEVEL_CSV_PATH = os.path.join(OUTDIR, "top_edges_distant_level.csv")
TOP_GLOBAL_LEVEL_CSV_PATH = os.path.join(OUTDIR, "top_edges_global_level.csv")

TOP_TOUCHES_LAG0_CSV_PATH = os.path.join(OUTDIR, "top_edges_touches_lag0.csv")

TOP_SAME_BASE_FEATURE_CSV_PATH = os.path.join(OUTDIR, "top_edges_same_base_feature.csv")
TOP_CROSS_BASE_FEATURE_CSV_PATH = os.path.join(OUTDIR, "top_edges_cross_base_feature.csv")

TOP_SAME_SIDE_CSV_PATH = os.path.join(OUTDIR, "top_edges_same_side_family.csv")
TOP_CROSS_SIDE_CSV_PATH = os.path.join(OUTDIR, "top_edges_cross_side_family.csv")

TOP_SAME_FEATURE_TYPE_CSV_PATH = os.path.join(OUTDIR, "top_edges_same_feature_type.csv")
TOP_CROSS_FEATURE_TYPE_CSV_PATH = os.path.join(OUTDIR, "top_edges_cross_feature_type.csv")

TOP_FEATURE_PAIR_DIR = os.path.join(OUTDIR, "by_feature_pair")
TOP_SIDE_PAIR_DIR = os.path.join(OUTDIR, "by_side_pair")
TOP_FEATURE_TYPE_PAIR_DIR = os.path.join(OUTDIR, "by_feature_type_pair")
os.makedirs(TOP_FEATURE_PAIR_DIR, exist_ok=True)
os.makedirs(TOP_SIDE_PAIR_DIR, exist_ok=True)
os.makedirs(TOP_FEATURE_TYPE_PAIR_DIR, exist_ok=True)

SUMMARY_JSON_PATH = os.path.join(OUTDIR, "lob_state_edge_rankings_summary.json")
META_JSON_PATH = os.path.join(OUTDIR, "lob_state_edge_rankings_metadata.json")


# ============================================================
# UTILS
# ============================================================

def clear_memory():
    """
    Trigger Python garbage collection.
    """
    gc.collect()


def top_k_edges(
    df: pd.DataFrame,
    k: int,
    sort_col: str = "abs_precision_weight"
) -> pd.DataFrame:
    """
    Return the top-k edges sorted by descending absolute weight.
    """
    if df.empty:
        return df.copy()

    out = df.sort_values(
        by=[sort_col, "source", "target"],
        ascending=[False, True, True],
    ).head(k).copy()

    out.reset_index(drop=True, inplace=True)
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    return out


def save_topk(df: pd.DataFrame, path: str, k: int) -> int:
    """
    Save the top-k ranking to CSV and return the number of saved rows.
    """
    top_df = top_k_edges(df, k=k, sort_col="abs_precision_weight")
    top_df.to_csv(path, index=False)
    return int(len(top_df))


def safe_filename(label: str) -> str:
    """
    Convert an arbitrary label into a safe filename fragment.
    """
    return (
        str(label)
        .replace("|", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
    )


def summarize_subset(df: pd.DataFrame) -> dict:
    """
    Compute base_variablesc descriptive statistics for an edge subset.
    """
    if df.empty:
        return {
            "n_edges": 0,
            "mean_abs_precision_weight": None,
            "median_abs_precision_weight": None,
            "max_abs_precision_weight": None,
            "mean_abs_partial_corr_weight": None,
            "median_abs_partial_corr_weight": None,
            "max_abs_partial_corr_weight": None,
        }

    return {
        "n_edges": int(len(df)),
        "mean_abs_precision_weight": float(df["abs_precision_weight"].mean()),
        "median_abs_precision_weight": float(df["abs_precision_weight"].median()),
        "max_abs_precision_weight": float(df["abs_precision_weight"].max()),
        "mean_abs_partial_corr_weight": float(df["abs_partial_corr_weight"].mean()),
        "median_abs_partial_corr_weight": float(df["abs_partial_corr_weight"].median()),
        "max_abs_partial_corr_weight": float(df["abs_partial_corr_weight"].max()),
    }


# ============================================================
# LOAD PARSED EDGE LIST
# ============================================================

print("Loading parsed edge list from MODULE 5A...")

if not os.path.exists(PARSED_EDGE_LIST_CSV_PATH):
    raise FileNotFoundError(f"Missing parsed edge list: {PARSED_EDGE_LIST_CSV_PATH}")

parsed_df = pd.read_csv(PARSED_EDGE_LIST_CSV_PATH)

print(f"Loaded parsed edge list shape : {parsed_df.shape}")


# ============================================================
# BASIC VALIDATION
# ============================================================

required_cols = {
    "source",
    "target",
    "precision_weight",
    "abs_precision_weight",
    "partial_corr_weight",
    "abs_partial_corr_weight",
    "sign",
    "src_base_feature",
    "tgt_base_feature",
    "src_feature_family",
    "tgt_feature_family",
    "src_side_family",
    "tgt_side_family",
    "src_feature_type",
    "tgt_feature_type",
    "src_level",
    "tgt_level",
    "src_lag",
    "tgt_lag",
    "level_relation",
    "lag_relation",
    "touches_lag0",
    "feature_pair",
    "side_pair",
    "feature_type_pair",
    "same_feature_type",
}

missing_cols = required_cols - set(parsed_df.columns)
if missing_cols:
    raise ValueError(f"Missing required parsed edge columns: {sorted(missing_cols)}")

if parsed_df.empty:
    raise ValueError("Parsed edge list is empty. Nothing to rank.")

print("\nInput validation passed.")
print(f"Number of parsed edges = {len(parsed_df)}")


# ============================================================
# DERIVED FLAGS
# ============================================================

parsed_df["same_base_feature"] = parsed_df["src_base_feature"] == parsed_df["tgt_base_feature"]
parsed_df["same_side_family"] = parsed_df["src_side_family"] == parsed_df["tgt_side_family"]


# ============================================================
# GLOBAL RANKINGS
# ============================================================

print("\nBuilding global edge rankings...")

n_top_overall = save_topk(parsed_df, TOP_OVERALL_CSV_PATH, TOP_K)
n_top_intra_lag = save_topk(
    parsed_df[parsed_df["lag_relation"] == "intra_lag"],
    TOP_INTRA_LAG_CSV_PATH,
    TOP_K,
)
n_top_cross_lag = save_topk(
    parsed_df[parsed_df["lag_relation"] != "intra_lag"],
    TOP_CROSS_LAG_CSV_PATH,
    TOP_K,
)

n_top_same_level = save_topk(
    parsed_df[parsed_df["level_relation"] == "same_level"],
    TOP_SAME_LEVEL_CSV_PATH,
    TOP_K,
)
n_top_adjacent_level = save_topk(
    parsed_df[parsed_df["level_relation"] == "adjacent_level"],
    TOP_ADJACENT_LEVEL_CSV_PATH,
    TOP_K,
)
n_top_distant_level = save_topk(
    parsed_df[parsed_df["level_relation"] == "distant_level"],
    TOP_DISTANT_LEVEL_CSV_PATH,
    TOP_K,
)
n_top_global_level = save_topk(
    parsed_df[parsed_df["level_relation"] == "global_level"],
    TOP_GLOBAL_LEVEL_CSV_PATH,
    TOP_K,
)

n_top_touches_lag0 = save_topk(
    parsed_df[parsed_df["touches_lag0"] == True],
    TOP_TOUCHES_LAG0_CSV_PATH,
    TOP_K,
)

n_top_same_base_feature = save_topk(
    parsed_df[parsed_df["same_base_feature"] == True],
    TOP_SAME_BASE_FEATURE_CSV_PATH,
    TOP_K,
)
n_top_cross_base_feature = save_topk(
    parsed_df[parsed_df["same_base_feature"] == False],
    TOP_CROSS_BASE_FEATURE_CSV_PATH,
    TOP_K,
)

n_top_same_side = save_topk(
    parsed_df[parsed_df["same_side_family"] == True],
    TOP_SAME_SIDE_CSV_PATH,
    TOP_K,
)
n_top_cross_side = save_topk(
    parsed_df[parsed_df["same_side_family"] == False],
    TOP_CROSS_SIDE_CSV_PATH,
    TOP_K,
)

n_top_same_feature_type = save_topk(
    parsed_df[parsed_df["same_feature_type"] == True],
    TOP_SAME_FEATURE_TYPE_CSV_PATH,
    TOP_K,
)
n_top_cross_feature_type = save_topk(
    parsed_df[parsed_df["same_feature_type"] == False],
    TOP_CROSS_FEATURE_TYPE_CSV_PATH,
    TOP_K,
)

print(f"Saved overall ranking         -> {os.path.abspath(TOP_OVERALL_CSV_PATH)}")
print(f"Saved intra-lag ranking       -> {os.path.abspath(TOP_INTRA_LAG_CSV_PATH)}")
print(f"Saved cross-lag ranking       -> {os.path.abspath(TOP_CROSS_LAG_CSV_PATH)}")
print(f"Saved same-level ranking      -> {os.path.abspath(TOP_SAME_LEVEL_CSV_PATH)}")
print(f"Saved adjacent-level rank     -> {os.path.abspath(TOP_ADJACENT_LEVEL_CSV_PATH)}")
print(f"Saved distant-level rank      -> {os.path.abspath(TOP_DISTANT_LEVEL_CSV_PATH)}")
print(f"Saved global-level rank       -> {os.path.abspath(TOP_GLOBAL_LEVEL_CSV_PATH)}")
print(f"Saved touches-lag0 rank       -> {os.path.abspath(TOP_TOUCHES_LAG0_CSV_PATH)}")
print(f"Saved same-base-feature       -> {os.path.abspath(TOP_SAME_BASE_FEATURE_CSV_PATH)}")
print(f"Saved cross-base-feature      -> {os.path.abspath(TOP_CROSS_BASE_FEATURE_CSV_PATH)}")
print(f"Saved same-side ranking       -> {os.path.abspath(TOP_SAME_SIDE_CSV_PATH)}")
print(f"Saved cross-side ranking      -> {os.path.abspath(TOP_CROSS_SIDE_CSV_PATH)}")
print(f"Saved same-feature-type rank  -> {os.path.abspath(TOP_SAME_FEATURE_TYPE_CSV_PATH)}")
print(f"Saved cross-feature-type rank -> {os.path.abspath(TOP_CROSS_FEATURE_TYPE_CSV_PATH)}")


# ============================================================
# RANKINGS BY FEATURE PAIR
# ============================================================

print("\nBuilding rankings by feature_pair...")

feature_pair_stats = {}

for pair_value in sorted(parsed_df["feature_pair"].dropna().unique()):
    df_sub = parsed_df[parsed_df["feature_pair"] == pair_value].copy()
    safe_name = safe_filename(pair_value)
    out_path = os.path.join(TOP_FEATURE_PAIR_DIR, f"top_edges_feature_pair_{safe_name}.csv")

    save_topk(df_sub, out_path, TOP_K)
    feature_pair_stats[pair_value] = summarize_subset(df_sub)

print(f"Saved rankings by feature pair   -> {os.path.abspath(TOP_FEATURE_PAIR_DIR)}")


# ============================================================
# RANKINGS BY SIDE PAIR
# ============================================================

print("\nBuilding rankings by side_pair...")

side_pair_stats = {}

for pair_value in sorted(parsed_df["side_pair"].dropna().unique()):
    df_sub = parsed_df[parsed_df["side_pair"] == pair_value].copy()
    safe_name = safe_filename(pair_value)
    out_path = os.path.join(TOP_SIDE_PAIR_DIR, f"top_edges_side_pair_{safe_name}.csv")

    save_topk(df_sub, out_path, TOP_K)
    side_pair_stats[pair_value] = summarize_subset(df_sub)

print(f"Saved rankings by side pair      -> {os.path.abspath(TOP_SIDE_PAIR_DIR)}")


# ============================================================
# RANKINGS BY FEATURE TYPE PAIR
# ============================================================

print("\nBuilding rankings by feature_type_pair...")

feature_type_pair_stats = {}

for pair_value in sorted(parsed_df["feature_type_pair"].dropna().unique()):
    df_sub = parsed_df[parsed_df["feature_type_pair"] == pair_value].copy()
    safe_name = safe_filename(pair_value)
    out_path = os.path.join(TOP_FEATURE_TYPE_PAIR_DIR, f"top_edges_feature_type_pair_{safe_name}.csv")

    save_topk(df_sub, out_path, TOP_K)
    feature_type_pair_stats[pair_value] = summarize_subset(df_sub)

print(f"Saved rankings by feature type   -> {os.path.abspath(TOP_FEATURE_TYPE_PAIR_DIR)}")


# ============================================================
# SUMMARY
# ============================================================

summary = {
    "top_k": int(TOP_K),
    "n_total_edges": int(len(parsed_df)),
    "global_rankings": {
        "overall": {
            "saved_rows": int(n_top_overall),
            **summarize_subset(parsed_df),
        },
        "intra_lag": {
            "saved_rows": int(n_top_intra_lag),
            **summarize_subset(parsed_df[parsed_df["lag_relation"] == "intra_lag"]),
        },
        "cross_lag": {
            "saved_rows": int(n_top_cross_lag),
            **summarize_subset(parsed_df[parsed_df["lag_relation"] != "intra_lag"]),
        },
        "same_level": {
            "saved_rows": int(n_top_same_level),
            **summarize_subset(parsed_df[parsed_df["level_relation"] == "same_level"]),
        },
        "adjacent_level": {
            "saved_rows": int(n_top_adjacent_level),
            **summarize_subset(parsed_df[parsed_df["level_relation"] == "adjacent_level"]),
        },
        "distant_level": {
            "saved_rows": int(n_top_distant_level),
            **summarize_subset(parsed_df[parsed_df["level_relation"] == "distant_level"]),
        },
        "global_level": {
            "saved_rows": int(n_top_global_level),
            **summarize_subset(parsed_df[parsed_df["level_relation"] == "global_level"]),
        },
        "touches_lag0": {
            "saved_rows": int(n_top_touches_lag0),
            **summarize_subset(parsed_df[parsed_df["touches_lag0"] == True]),
        },
        "same_base_feature": {
            "saved_rows": int(n_top_same_base_feature),
            **summarize_subset(parsed_df[parsed_df["same_base_feature"] == True]),
        },
        "cross_base_feature": {
            "saved_rows": int(n_top_cross_base_feature),
            **summarize_subset(parsed_df[parsed_df["same_base_feature"] == False]),
        },
        "same_side_family": {
            "saved_rows": int(n_top_same_side),
            **summarize_subset(parsed_df[parsed_df["same_side_family"] == True]),
        },
        "cross_side_family": {
            "saved_rows": int(n_top_cross_side),
            **summarize_subset(parsed_df[parsed_df["same_side_family"] == False]),
        },
        "same_feature_type": {
            "saved_rows": int(n_top_same_feature_type),
            **summarize_subset(parsed_df[parsed_df["same_feature_type"] == True]),
        },
        "cross_feature_type": {
            "saved_rows": int(n_top_cross_feature_type),
            **summarize_subset(parsed_df[parsed_df["same_feature_type"] == False]),
        },
    },
    "feature_pair_rankings": feature_pair_stats,
    "side_pair_rankings": side_pair_stats,
    "feature_type_pair_rankings": feature_type_pair_stats,
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved summary               -> {os.path.abspath(SUMMARY_JSON_PATH)}")


# ============================================================
# METADATA
# ============================================================

meta = {
    "module": "MODULE 5B - STATE GRAPH - INTERPRETABLE EDGE RANKINGS",
    "input_parsed_edge_list_csv": PARSED_EDGE_LIST_CSV_PATH,
    "top_k": int(TOP_K),
    "top_overall_csv": TOP_OVERALL_CSV_PATH,
    "top_intra_lag_csv": TOP_INTRA_LAG_CSV_PATH,
    "top_cross_lag_csv": TOP_CROSS_LAG_CSV_PATH,
    "top_same_level_csv": TOP_SAME_LEVEL_CSV_PATH,
    "top_adjacent_level_csv": TOP_ADJACENT_LEVEL_CSV_PATH,
    "top_distant_level_csv": TOP_DISTANT_LEVEL_CSV_PATH,
    "top_global_level_csv": TOP_GLOBAL_LEVEL_CSV_PATH,
    "top_touches_lag0_csv": TOP_TOUCHES_LAG0_CSV_PATH,
    "top_same_base_feature_csv": TOP_SAME_BASE_FEATURE_CSV_PATH,
    "top_cross_base_feature_csv": TOP_CROSS_BASE_FEATURE_CSV_PATH,
    "top_same_side_family_csv": TOP_SAME_SIDE_CSV_PATH,
    "top_cross_side_family_csv": TOP_CROSS_SIDE_CSV_PATH,
    "top_same_feature_type_csv": TOP_SAME_FEATURE_TYPE_CSV_PATH,
    "top_cross_feature_type_csv": TOP_CROSS_FEATURE_TYPE_CSV_PATH,
    "top_feature_pair_dir": TOP_FEATURE_PAIR_DIR,
    "top_side_pair_dir": TOP_SIDE_PAIR_DIR,
    "top_feature_type_pair_dir": TOP_FEATURE_TYPE_PAIR_DIR,
    "summary_json": SUMMARY_JSON_PATH,
}

with open(META_JSON_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata              -> {os.path.abspath(META_JSON_PATH)}")


# ============================================================
# QUICK CHECKS
# ============================================================

print("\nQuick checks:")
print(f"Total parsed edges            : {len(parsed_df)}")
print(f"Top overall saved             : {n_top_overall}")
print(f"Top intra-lag saved           : {n_top_intra_lag}")
print(f"Top cross-lag saved           : {n_top_cross_lag}")
print(f"Top same-level saved          : {n_top_same_level}")
print(f"Top adjacent-level saved      : {n_top_adjacent_level}")
print(f"Top distant-level saved       : {n_top_distant_level}")
print(f"Top global-level saved        : {n_top_global_level}")
print(f"Top touches-lag0 saved        : {n_top_touches_lag0}")
print(f"Top same-base-feature saved   : {n_top_same_base_feature}")
print(f"Top cross-base-feature saved  : {n_top_cross_base_feature}")
print(f"Top same-side-family saved    : {n_top_same_side}")
print(f"Top cross-side-family saved   : {n_top_cross_side}")
print(f"Top same-feature-type saved   : {n_top_same_feature_type}")
print(f"Top cross-feature-type saved  : {n_top_cross_feature_type}")

print("\nDONE: interpretable edge rankings completed successfully.")

In [ ]:
# ============================================================
# MODULE 6
# STATE GRAPH - FINAL SCREEN REPORT
# Read saved CSV/JSON files and print a structured analysis report
# Compatible with both:
#   - per-level features
#   - global semantic features
# ============================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = "./glasso_cache_v4"

FINAL_GRAPH_DIR = os.path.join(BASE_DIR, "state_graph_final")
EDGE_ANALYSIS_DIR = os.path.join(BASE_DIR, "state_graph_edge_analysis")
EDGE_RANKINGS_DIR = os.path.join(BASE_DIR, "state_graph_edge_rankings")

# ---- Module 4 outputs
GLASSO_SUMMARY_JSON_PATH = os.path.join(FINAL_GRAPH_DIR, "lob_state_glasso_summary.json")
TOP_EDGES_CSV_PATH = os.path.join(FINAL_GRAPH_DIR, "lob_state_glasso_top_edges.csv")

# ---- Module 5A outputs
EDGE_ANALYSIS_SUMMARY_JSON_PATH = os.path.join(EDGE_ANALYSIS_DIR, "lob_state_edge_analysis_summary.json")
SUMMARY_BY_LAG_TYPE_CSV_PATH = os.path.join(EDGE_ANALYSIS_DIR, "summary_by_lag_type.csv")
SUMMARY_BY_LEVEL_REL_CSV_PATH = os.path.join(EDGE_ANALYSIS_DIR, "summary_by_level_relation.csv")
SUMMARY_BY_FEATURE_PAIR_CSV_PATH = os.path.join(EDGE_ANALYSIS_DIR, "summary_by_feature_pair.csv")
SUMMARY_BY_SIDE_PAIR_CSV_PATH = os.path.join(EDGE_ANALYSIS_DIR, "summary_by_side_pair.csv")
SUMMARY_BY_TOUCHES_LAG0_CSV_PATH = os.path.join(EDGE_ANALYSIS_DIR, "summary_by_touches_lag0.csv")
TOP_EDGES_WITH_CATEGORIES_CSV_PATH = os.path.join(EDGE_ANALYSIS_DIR, "top_edges_by_abs_precision_with_categories.csv")

# ---- Module 5B outputs
EDGE_RANKINGS_SUMMARY_JSON_PATH = os.path.join(EDGE_RANKINGS_DIR, "lob_state_edge_rankings_summary.json")
TOP_OVERALL_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_overall.csv")
TOP_INTRA_LAG_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_intra_lag.csv")
TOP_CROSS_LAG_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_cross_lag.csv")
TOP_SAME_LEVEL_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_same_level.csv")
TOP_ADJACENT_LEVEL_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_adjacent_level.csv")
TOP_DISTANT_LEVEL_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_distant_level.csv")
TOP_GLOBAL_LEVEL_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_global_level.csv")
TOP_TOUCHES_LAG0_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_touches_lag0.csv")
TOP_SAME_BASE_FEATURE_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_same_base_feature.csv")
TOP_CROSS_BASE_FEATURE_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_cross_base_feature.csv")
TOP_SAME_SIDE_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_same_side_family.csv")
TOP_CROSS_SIDE_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_cross_side_family.csv")
TOP_SAME_FEATURE_TYPE_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_same_feature_type.csv")
TOP_CROSS_FEATURE_TYPE_CSV_PATH = os.path.join(EDGE_RANKINGS_DIR, "top_edges_cross_feature_type.csv")

# ---- Display settings
TOP_N_PREVIEW = 10
FLOAT_FMT = "{:.6f}"


# ============================================================
# UTILS
# ============================================================

def print_rule(char: str = "=", width: int = 76):
    print(char * width)


def print_section(title: str):
    print()
    print_rule("=")
    print(title)
    print_rule("=")


def print_subsection(title: str):
    print()
    print_rule("-")
    print(title)
    print_rule("-")


def load_json(path: str) -> dict:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing JSON file: {path}")
    with open(path, "r") as f:
        return json.load(f)


def load_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing CSV file: {path}")
    return pd.read_csv(path)


def fmt_value(x):
    if x is None:
        return "None"
    if isinstance(x, (float, np.floating)):
        return FLOAT_FMT.format(float(x))
    return str(x)


def pretty_print_kv(d: dict, order: list[str] | None = None):
    keys = order if order is not None else list(d.keys())
    keys = [k for k in keys if k in d]

    if not keys:
        print("[no keys]")
        return

    max_len = max(len(str(k)) for k in keys)
    for k in keys:
        print(f"{k:<{max_len}} : {fmt_value(d[k])}")


def print_df(df: pd.DataFrame, n: int = 10, columns: list[str] | None = None):
    if df.empty:
        print("[empty]")
        return

    out = df.copy()

    if columns is not None:
        cols = [c for c in columns if c in out.columns]
        out = out[cols]

    print(out.head(n).to_string(index=False))


def summarize_top_edges(df: pd.DataFrame, title: str, n: int = 10):
    print_subsection(title)
    cols = [
        "rank",
        "source",
        "target",
        "precision_weight",
        "abs_precision_weight",
        "partial_corr_weight",
        "sign",
        "lag_relation",
        "level_relation",
        "feature_pair",
        "side_pair",
        "feature_type_pair",
    ]
    print_df(df, n=n, columns=cols)


def get_global_ranking_entry(summary: dict, key: str) -> dict:
    return summary.get("global_rankings", {}).get(key, {})


def print_ranking_summary_block(summary_json: dict):
    block = {
        "top_k": summary_json.get("top_k"),
        "n_total_edges": summary_json.get("n_total_edges"),

        "overall_n_edges": get_global_ranking_entry(summary_json, "overall").get("n_edges"),
        "intra_lag_n_edges": get_global_ranking_entry(summary_json, "intra_lag").get("n_edges"),
        "cross_lag_n_edges": get_global_ranking_entry(summary_json, "cross_lag").get("n_edges"),

        "same_level_n_edges": get_global_ranking_entry(summary_json, "same_level").get("n_edges"),
        "adjacent_level_n_edges": get_global_ranking_entry(summary_json, "adjacent_level").get("n_edges"),
        "distant_level_n_edges": get_global_ranking_entry(summary_json, "distant_level").get("n_edges"),
        "global_level_n_edges": get_global_ranking_entry(summary_json, "global_level").get("n_edges"),

        "touches_lag0_n_edges": get_global_ranking_entry(summary_json, "touches_lag0").get("n_edges"),

        "same_base_feature_n_edges": get_global_ranking_entry(summary_json, "same_base_feature").get("n_edges"),
        "cross_base_feature_n_edges": get_global_ranking_entry(summary_json, "cross_base_feature").get("n_edges"),

        "same_side_family_n_edges": get_global_ranking_entry(summary_json, "same_side_family").get("n_edges"),
        "cross_side_family_n_edges": get_global_ranking_entry(summary_json, "cross_side_family").get("n_edges"),

        "same_feature_type_n_edges": get_global_ranking_entry(summary_json, "same_feature_type").get("n_edges"),
        "cross_feature_type_n_edges": get_global_ranking_entry(summary_json, "cross_feature_type").get("n_edges"),
    }
    pretty_print_kv(block)


# ============================================================
# LOAD ALL REQUIRED FILES
# ============================================================

glasso_summary = load_json(GLASSO_SUMMARY_JSON_PATH)
edge_analysis_summary = load_json(EDGE_ANALYSIS_SUMMARY_JSON_PATH)
edge_rankings_summary = load_json(EDGE_RANKINGS_SUMMARY_JSON_PATH)

summary_by_lag_type = load_csv(SUMMARY_BY_LAG_TYPE_CSV_PATH)
summary_by_level_rel = load_csv(SUMMARY_BY_LEVEL_REL_CSV_PATH)
summary_by_feature_pair = load_csv(SUMMARY_BY_FEATURE_PAIR_CSV_PATH)
summary_by_side_pair = load_csv(SUMMARY_BY_SIDE_PAIR_CSV_PATH)
summary_by_touches_lag0 = load_csv(SUMMARY_BY_TOUCHES_LAG0_CSV_PATH)

top_edges_module4 = load_csv(TOP_EDGES_CSV_PATH)
top_edges_with_categories = load_csv(TOP_EDGES_WITH_CATEGORIES_CSV_PATH)

top_overall = load_csv(TOP_OVERALL_CSV_PATH)
top_intra_lag = load_csv(TOP_INTRA_LAG_CSV_PATH)
top_cross_lag = load_csv(TOP_CROSS_LAG_CSV_PATH)
top_same_level = load_csv(TOP_SAME_LEVEL_CSV_PATH)
top_adjacent_level = load_csv(TOP_ADJACENT_LEVEL_CSV_PATH)
top_distant_level = load_csv(TOP_DISTANT_LEVEL_CSV_PATH)
top_global_level = load_csv(TOP_GLOBAL_LEVEL_CSV_PATH)
top_touches_lag0 = load_csv(TOP_TOUCHES_LAG0_CSV_PATH)
top_same_base_feature = load_csv(TOP_SAME_BASE_FEATURE_CSV_PATH)
top_cross_base_feature = load_csv(TOP_CROSS_BASE_FEATURE_CSV_PATH)
top_same_side = load_csv(TOP_SAME_SIDE_CSV_PATH)
top_cross_side = load_csv(TOP_CROSS_SIDE_CSV_PATH)
top_same_feature_type = load_csv(TOP_SAME_FEATURE_TYPE_CSV_PATH)
top_cross_feature_type = load_csv(TOP_CROSS_FEATURE_TYPE_CSV_PATH)


# ============================================================
# REPORT
# ============================================================

print_section("STATE GRAPH - FINAL ANALYSIS REPORT")

print_subsection("1) FINAL GRAPH OVERVIEW (MODULE 4)")
pretty_print_kv(
    glasso_summary,
    order=[
        "best_alpha",
        "fit_time_sec",
        "n_samples",
        "n_features",
        "n_possible_edges",
        "n_edges",
        "graph_density",
        "precision_offdiag_nnz",
        "adjacency_undirected_edges",
        "diag_mean_covariance",
        "diag_mean_precision",
        "diag_mean_partial_corr",
        "symmetry_error_covariance",
        "symmetry_error_precision",
        "symmetry_error_partial_corr",
        "symmetry_error_adjacency",
    ]
)

print_subsection("2) STRUCTURAL EDGE SUMMARY (MODULE 5A)")
pretty_print_kv(
    edge_analysis_summary,
    order=[
        "n_edges",
        "n_intra_lag",
        "share_intra_lag",
        "n_cross_lag",
        "share_cross_lag",
        "n_same_level",
        "share_same_level",
        "n_adjacent_level",
        "share_adjacent_level",
        "n_distant_level",
        "share_distant_level",
        "n_global_level",
        "share_global_level",
        "n_touches_lag0",
        "share_touches_lag0",
        "n_both_lag0",
        "share_both_lag0",
        "n_crosses_time",
        "share_crosses_time",
        "mean_abs_precision_weight",
        "median_abs_precision_weight",
        "max_abs_precision_weight",
        "mean_abs_partial_corr_weight",
        "median_abs_partial_corr_weight",
        "max_abs_partial_corr_weight",
    ]
)

print_subsection("3) SUMMARY BY LAG TYPE")
print_df(
    summary_by_lag_type,
    n=len(summary_by_lag_type),
    columns=[
        "lag_relation",
        "n_edges",
        "share_edges",
        "mean_abs_precision_weight",
        "median_abs_precision_weight",
        "max_abs_precision_weight",
        "mean_abs_partial_corr_weight",
        "median_abs_partial_corr_weight",
        "max_abs_partial_corr_weight",
    ]
)

print_subsection("4) SUMMARY BY LEVEL RELATION")
print_df(
    summary_by_level_rel,
    n=len(summary_by_level_rel),
    columns=[
        "level_relation",
        "n_edges",
        "share_edges",
        "mean_abs_precision_weight",
        "median_abs_precision_weight",
        "max_abs_precision_weight",
        "mean_abs_partial_corr_weight",
        "median_abs_partial_corr_weight",
        "max_abs_partial_corr_weight",
    ]
)

print_subsection("5) SUMMARY BY FEATURE PAIR")
print_df(
    summary_by_feature_pair,
    n=len(summary_by_feature_pair),
    columns=[
        "feature_pair",
        "n_edges",
        "share_edges",
        "mean_abs_precision_weight",
        "median_abs_precision_weight",
        "max_abs_precision_weight",
        "mean_abs_partial_corr_weight",
        "median_abs_partial_corr_weight",
        "max_abs_partial_corr_weight",
    ]
)

print_subsection("6) SUMMARY BY SIDE PAIR")
print_df(
    summary_by_side_pair,
    n=len(summary_by_side_pair),
    columns=[
        "side_pair",
        "n_edges",
        "share_edges",
        "mean_abs_precision_weight",
        "median_abs_precision_weight",
        "max_abs_precision_weight",
        "mean_abs_partial_corr_weight",
        "median_abs_partial_corr_weight",
        "max_abs_partial_corr_weight",
    ]
)

print_subsection("7) SUMMARY BY TOUCHES LAG 0")
print_df(
    summary_by_touches_lag0,
    n=len(summary_by_touches_lag0),
    columns=[
        "touches_lag0",
        "n_edges",
        "share_edges",
        "mean_abs_precision_weight",
        "median_abs_precision_weight",
        "max_abs_precision_weight",
        "mean_abs_partial_corr_weight",
        "median_abs_partial_corr_weight",
        "max_abs_partial_corr_weight",
    ]
)

print_subsection("8) MODULE 5B GLOBAL RANKING SUMMARY")
print_ranking_summary_block(edge_rankings_summary)

print_subsection("9) TOP EDGES FROM MODULE 4 (RAW EXPORT)")
print_df(
    top_edges_module4,
    n=TOP_N_PREVIEW,
    columns=[
        "source",
        "target",
        "precision_weight",
        "abs_precision_weight",
        "partial_corr_weight",
        "abs_partial_corr_weight",
        "sign",
    ]
)

summarize_top_edges(top_overall, "10) TOP OVERALL EDGES (MODULE 5B)", n=TOP_N_PREVIEW)
summarize_top_edges(top_cross_lag, "11) TOP CROSS-LAG EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_intra_lag, "12) TOP INTRA-LAG EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_same_level, "13) TOP SAME-LEVEL EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_adjacent_level, "14) TOP ADJACENT-LEVEL EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_distant_level, "15) TOP DISTANT-LEVEL EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_global_level, "16) TOP GLOBAL-LEVEL EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_touches_lag0, "17) TOP EDGES TOUCHING LAG 0", n=TOP_N_PREVIEW)
summarize_top_edges(top_same_base_feature, "18) TOP SAME-BASE-FEATURE EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_cross_base_feature, "19) TOP CROSS-BASE-FEATURE EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_same_side, "20) TOP SAME-SIDE-FAMILY EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_cross_side, "21) TOP CROSS-SIDE-FAMILY EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_same_feature_type, "22) TOP SAME-FEATURE-TYPE EDGES", n=TOP_N_PREVIEW)
summarize_top_edges(top_cross_feature_type, "23) TOP CROSS-FEATURE-TYPE EDGES", n=TOP_N_PREVIEW)

print_subsection("24) TOP EDGES WITH FULL CATEGORIES (MODULE 5A)")
print_df(
    top_edges_with_categories,
    n=TOP_N_PREVIEW,
    columns=[
        "source",
        "target",
        "abs_precision_weight",
        "partial_corr_weight",
        "lag_relation",
        "level_relation",
        "feature_pair",
        "side_pair",
        "feature_type_pair",
        "src_feature_type",
        "tgt_feature_type",
    ]
)

print_subsection("25) COMPACT INTERPRETATION CHECKLIST")

n_edges = glasso_summary.get("n_edges", 0)
density = glasso_summary.get("graph_density", 0.0)
share_cross_lag = edge_analysis_summary.get("share_cross_lag", 0.0)
share_same_level = edge_analysis_summary.get("share_same_level", 0.0)
share_global_level = edge_analysis_summary.get("share_global_level", 0.0)

print(f"- Graph density                 : {density:.4f}")
if density < 0.02:
    print("  Interpretation               : very sparse graph")
elif density < 0.15:
    print("  Interpretation               : reasonably sparse graph")
else:
    print("  Interpretation               : relatively dense graph")

print(f"- Share of cross-lag edges     : {share_cross_lag:.4f}")
if share_cross_lag > 0.60:
    print("  Interpretation               : temporal dependencies dominate")
elif share_cross_lag > 0.35:
    print("  Interpretation               : mixed temporal/static structure")
else:
    print("  Interpretation               : mostly intra-snapshot structure")

print(f"- Share of same-level edges    : {share_same_level:.4f}")
if share_same_level > 0.50:
    print("  Interpretation               : interactions are concentrated within the same level")
elif share_same_level > 0.25:
    print("  Interpretation               : same-level and nearby-level interactions both matter")
else:
    print("  Interpretation               : cross-level interactions dominate")

print(f"- Share of global-level edges  : {share_global_level:.4f}")
if share_global_level > 0.20:
    print("  Interpretation               : global semantic features are materially involved")
elif share_global_level > 0.00:
    print("  Interpretation               : global semantic features appear, but are not dominant")
else:
    print("  Interpretation               : no global semantic feature edges detected")

print(f"- Total undirected edges       : {n_edges}")
print("  Interpretation               : use this together with rankings to judge graph informativeness")

print()
print_rule("=")
print("END OF REPORT")
print_rule("=")

In [ ]:
# ============================================================
# PLOT 3D GRAPH WITH NETWORKX + PLOTLY (SPRING LAYOUT)
# input : final edge list from MODULE 4
# output: interactive 3D graph
# ============================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go


# ============================================================
# CONFIG
# ============================================================

FINAL_GRAPH_DIR = "./glasso_cache_v4_lag20/state_graph_final"

EDGE_LIST_CSV_PATH = os.path.join(FINAL_GRAPH_DIR, "lob_state_glasso_edge_list.csv")

# ---- graph filtering
TOP_K_EDGES = 10000000          # use top-k strongest edges for visualization
MIN_ABS_WEIGHT = None      
REMOVE_ISOLATES = False

# ---- spring layout
SPRING_K = 0.9
SPRING_ITERATIONS = 300
SPRING_SEED = 42
DIM = 3

# ---- node sizing
BASE_NODE_SIZE = 6
DEGREE_SIZE_SCALE = 2.5

# ---- edge styling
EDGE_WIDTH_MIN = 1.0
EDGE_WIDTH_MAX = 6.0
EDGE_OPACITY = 0.35

# ---- plotting
FIG_WIDTH = 1200
FIG_HEIGHT = 900
TITLE = "3D State Graph (Spring Layout)"

OUTPUT_HTML = os.path.join(FINAL_GRAPH_DIR, "state_graph_3d.html")


# ============================================================
# LOAD EDGE LIST
# ============================================================

print("Loading edge list...")

if not os.path.exists(EDGE_LIST_CSV_PATH):
    raise FileNotFoundError(f"Missing edge list: {EDGE_LIST_CSV_PATH}")

edge_df = pd.read_csv(EDGE_LIST_CSV_PATH)

required_cols = {
    "source",
    "target",
    "precision_weight",
    "abs_precision_weight",
    "partial_corr_weight",
    "abs_partial_corr_weight",
    "sign",
}

missing_cols = required_cols - set(edge_df.columns)
if missing_cols:
    raise ValueError(f"Missing required edge columns: {sorted(missing_cols)}")

if edge_df.empty:
    raise ValueError("Edge list is empty.")

print(f"Loaded edge list shape: {edge_df.shape}")


# ============================================================
# FILTER EDGES FOR VISUALIZATION
# ============================================================

print("\nFiltering edges for visualization...")

plot_df = edge_df.copy()

if MIN_ABS_WEIGHT is not None:
    plot_df = plot_df[plot_df["abs_precision_weight"] >= MIN_ABS_WEIGHT].copy()

plot_df = plot_df.sort_values(
    by=["abs_precision_weight", "source", "target"],
    ascending=[False, True, True],
).copy()

if TOP_K_EDGES is not None:
    plot_df = plot_df.head(TOP_K_EDGES).copy()

plot_df.reset_index(drop=True, inplace=True)

if plot_df.empty:
    raise ValueError("No edges left after filtering.")

print(f"Edges kept for plot: {len(plot_df)}")


# ============================================================
# BUILD NETWORKX GRAPH
# ============================================================

print("\nBuilding NetworkX graph...")

G = nx.Graph()

for _, row in plot_df.iterrows():
    G.add_edge(
        row["source"],
        row["target"],
        precision_weight=float(row["precision_weight"]),
        abs_precision_weight=float(row["abs_precision_weight"]),
        partial_corr_weight=float(row["partial_corr_weight"]),
        abs_partial_corr_weight=float(row["abs_partial_corr_weight"]),
        sign=row["sign"],
    )

if REMOVE_ISOLATES:
    isolates = list(nx.isolates(G))
    if len(isolates) > 0:
        G.remove_nodes_from(isolates)

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()

if n_nodes == 0 or n_edges == 0:
    raise ValueError("Graph is empty after isolate removal.")

print(f"Graph nodes: {n_nodes}")
print(f"Graph edges: {n_edges}")


# ============================================================
# 3D SPRING LAYOUT
# ============================================================

print("\nComputing 3D spring layout...")

pos = nx.spring_layout(
    G,
    dim=DIM,
    seed=SPRING_SEED,
    k=SPRING_K,
    iterations=SPRING_ITERATIONS,
    weight="abs_precision_weight",
)

print("Layout computed successfully.")


# ============================================================
# NODE METADATA
# ============================================================

print("\nPreparing node metadata...")

node_degree = dict(G.degree())
node_strength = {
    node: sum(abs(G[node][nbr]["precision_weight"]) for nbr in G.neighbors(node))
    for node in G.nodes()
}

degrees = np.array([node_degree[n] for n in G.nodes()], dtype=float)
strengths = np.array([node_strength[n] for n in G.nodes()], dtype=float)

if degrees.max() > degrees.min():
    node_sizes = BASE_NODE_SIZE + DEGREE_SIZE_SCALE * (degrees - degrees.min()) / (degrees.max() - degrees.min()) * 10.0
else:
    node_sizes = np.full_like(degrees, BASE_NODE_SIZE + 5.0)

node_x = []
node_y = []
node_z = []
node_text = []
node_color = []

for idx, node in enumerate(G.nodes()):
    x, y, z = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_z.append(z)

    degree_val = node_degree[node]
    strength_val = node_strength[node]

    node_text.append(
        f"node: {node}<br>"
        f"degree: {degree_val}<br>"
        f"strength(abs precision): {strength_val:.6f}"
    )

    node_color.append(degree_val)


# ============================================================
# EDGE TRACES
# Build one trace per sign to color positive/negative separately
# ============================================================

print("\nPreparing edge traces...")

abs_weights = np.array(
    [G[u][v]["abs_precision_weight"] for u, v in G.edges()],
    dtype=float
)

w_min = abs_weights.min()
w_max = abs_weights.max()

def scale_width(w: float) -> float:
    if w_max == w_min:
        return (EDGE_WIDTH_MIN + EDGE_WIDTH_MAX) / 2.0
    alpha = (w - w_min) / (w_max - w_min)
    return EDGE_WIDTH_MIN + alpha * (EDGE_WIDTH_MAX - EDGE_WIDTH_MIN)

edge_traces = []

for sign_value, color_value in [("positive", "#1f77b4"), ("negative", "#d62728")]:
    sign_edges = [(u, v) for u, v in G.edges() if G[u][v]["sign"] == sign_value]

    if len(sign_edges) == 0:
        continue

    for u, v in sign_edges:
        x0, y0, z0 = pos[u]
        x1, y1, z1 = pos[v]

        abs_w = G[u][v]["abs_precision_weight"]
        p_w = G[u][v]["precision_weight"]
        pc_w = G[u][v]["partial_corr_weight"]

        edge_traces.append(
            go.Scatter3d(
                x=[x0, x1, None],
                y=[y0, y1, None],
                z=[z0, z1, None],
                mode="lines",
                line=dict(
                    color=color_value,
                    width=scale_width(abs_w),
                ),
                opacity=EDGE_OPACITY,
                hoverinfo="text",
                text=[
                    f"{u} ↔ {v}<br>"
                    f"sign: {sign_value}<br>"
                    f"precision_weight: {p_w:.6f}<br>"
                    f"abs_precision_weight: {abs_w:.6f}<br>"
                    f"partial_corr_weight: {pc_w:.6f}",
                    f"{u} ↔ {v}<br>"
                    f"sign: {sign_value}<br>"
                    f"precision_weight: {p_w:.6f}<br>"
                    f"abs_precision_weight: {abs_w:.6f}<br>"
                    f"partial_corr_weight: {pc_w:.6f}",
                    None
                ],
                showlegend=False,
            )
        )


# ============================================================
# NODE TRACE
# ============================================================

node_trace = go.Scatter3d(
    x=node_x,
    y=node_y,
    z=node_z,
    mode="markers+text",
    text=[n for n in G.nodes()],
    textposition="top center",
    hoverinfo="text",
    hovertext=node_text,
    marker=dict(
        size=node_sizes,
        color=node_color,
        colorscale="Viridis",
        colorbar=dict(title="Degree"),
        opacity=0.95,
        line=dict(width=0.5, color="black"),
    ),
    showlegend=False,
)


# ============================================================
# FIGURE
# ============================================================

print("\nRendering interactive Plotly figure...")

fig = go.Figure(data=edge_traces + [node_trace])

fig.update_layout(
    title=(
        f"{TITLE}<br>"
        f"<sup>nodes={n_nodes}, edges={n_edges}, "
        f"top_k_edges={TOP_K_EDGES}, spring_k={SPRING_K}, iterations={SPRING_ITERATIONS}</sup>"
    ),
    #width=FIG_WIDTH,
    #height=FIG_HEIGHT,
    autosize=True,
    margin=dict(l=0, r=0, b=0, t=60),
    scene=dict(
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
    ),
)

#fig.show()
fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn")
print(f"\nSaved HTML to: {OUTPUT_HTML}")

print("\nDONE: 3D graph rendered successfully.")

In [ ]:
# ============================================================
# PLOT 3D GRAPH WITH NETWORKX + PLOTLY
# NODE COLORS BY FEATURE FAMILY
# ============================================================

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go


# ============================================================
# CONFIG
# ============================================================

FINAL_GRAPH_DIR = "./glasso_cache_v4_lag20/state_graph_final"
EDGE_LIST_CSV_PATH = os.path.join(FINAL_GRAPH_DIR, "lob_state_glasso_edge_list.csv")

TOP_K_EDGES = 10000000        # number of edges to keep for visualization (after sorting by abs_precision_weight)
MIN_ABS_WEIGHT = None
REMOVE_ISOLATES = False

SPRING_K = 0.9
SPRING_ITERATIONS = 300
SPRING_SEED = 42
DIM = 3

BASE_NODE_SIZE = 6
DEGREE_SIZE_SCALE = 2.5

EDGE_WIDTH_MIN = 1.0
EDGE_WIDTH_MAX = 6.0
EDGE_OPACITY = 0.35

FIG_WIDTH = 1200
FIG_HEIGHT = 900
TITLE = "3D State Graph - Node Colors by Family"

OUTPUT_HTML = os.path.join(FINAL_GRAPH_DIR, "state_graph_3d_by_family.html")


# ============================================================
# FEATURE FAMILY PARSER
# ============================================================

def get_node_family(node: str) -> str:
    """
    Assign each node to an interpretable feature family.
    """

    if re.match(r"^askp_\d+_lag_\d+$", node):
        return "ask_price"

    if re.match(r"^bidp_\d+_lag_\d+$", node):
        return "bid_price"

    if re.match(r"^asks_\d+_lag_\d+$", node):
        return "ask_size"

    if re.match(r"^bids_\d+_lag_\d+$", node):
        return "bid_size"

    if re.match(r"^mid_price_lag_\d+$", node):
        return "mid_price"

    if re.match(r"^spread_lag_\d+$", node):
        return "spread"

    if re.match(r"^depth_ask_total_lag_\d+$", node):
        return "depth_ask_total"

    if re.match(r"^depth_bid_total_lag_\d+$", node):
        return "depth_bid_total"

    if re.match(r"^global_imbalance_lag_\d+$", node):
        return "global_imbalance"

    return "unknown"


FAMILY_COLORS = {
    "ask_price": "#d62728",
    "ask_size": "#ff9896",
    "bid_price": "#1f77b4",
    "bid_size": "#aec7e8",
    "mid_price": "#9467bd",
    "spread": "#8c564b",
    "depth_ask_total": "#ff7f0e",
    "depth_bid_total": "#2ca02c",
    "global_imbalance": "#bcbd22",
    "unknown": "#7f7f7f",
}


# ============================================================
# LOAD EDGE LIST
# ============================================================

print("Loading edge list...")

if not os.path.exists(EDGE_LIST_CSV_PATH):
    raise FileNotFoundError(f"Missing edge list: {EDGE_LIST_CSV_PATH}")

edge_df = pd.read_csv(EDGE_LIST_CSV_PATH)

required_cols = {
    "source",
    "target",
    "precision_weight",
    "abs_precision_weight",
    "partial_corr_weight",
    "abs_partial_corr_weight",
    "sign",
}

missing_cols = required_cols - set(edge_df.columns)
if missing_cols:
    raise ValueError(f"Missing required edge columns: {sorted(missing_cols)}")

if edge_df.empty:
    raise ValueError("Edge list is empty.")

print(f"Loaded edge list shape: {edge_df.shape}")


# ============================================================
# FILTER EDGES
# ============================================================

print("\nFiltering edges for visualization...")

plot_df = edge_df.copy()

if MIN_ABS_WEIGHT is not None:
    plot_df = plot_df[plot_df["abs_precision_weight"] >= MIN_ABS_WEIGHT].copy()

plot_df = plot_df.sort_values(
    by=["abs_precision_weight", "source", "target"],
    ascending=[False, True, True],
).copy()

if TOP_K_EDGES is not None:
    plot_df = plot_df.head(TOP_K_EDGES).copy()

plot_df.reset_index(drop=True, inplace=True)

if plot_df.empty:
    raise ValueError("No edges left after filtering.")

print(f"Edges kept for plot: {len(plot_df)}")


# ============================================================
# BUILD GRAPH
# ============================================================

print("\nBuilding NetworkX graph...")

G = nx.Graph()

for _, row in plot_df.iterrows():
    G.add_edge(
        row["source"],
        row["target"],
        precision_weight=float(row["precision_weight"]),
        abs_precision_weight=float(row["abs_precision_weight"]),
        partial_corr_weight=float(row["partial_corr_weight"]),
        abs_partial_corr_weight=float(row["abs_partial_corr_weight"]),
        sign=row["sign"],
    )

if REMOVE_ISOLATES:
    isolates = list(nx.isolates(G))
    if isolates:
        G.remove_nodes_from(isolates)

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()

if n_nodes == 0 or n_edges == 0:
    raise ValueError("Graph is empty after isolate removal.")

print(f"Graph nodes: {n_nodes}")
print(f"Graph edges: {n_edges}")


# ============================================================
# SPRING LAYOUT
# ============================================================

print("\nComputing 3D spring layout...")

pos = nx.spring_layout(
    G,
    dim=DIM,
    seed=SPRING_SEED,
    k=SPRING_K,
    iterations=SPRING_ITERATIONS,
    weight="abs_precision_weight",
)

print("Layout computed successfully.")


# ============================================================
# NODE METADATA
# ============================================================

print("\nPreparing node metadata...")

node_degree = dict(G.degree())
node_strength = {
    node: sum(abs(G[node][nbr]["precision_weight"]) for nbr in G.neighbors(node))
    for node in G.nodes()
}

degrees = np.array([node_degree[n] for n in G.nodes()], dtype=float)

if degrees.max() > degrees.min():
    node_sizes = BASE_NODE_SIZE + DEGREE_SIZE_SCALE * (
        degrees - degrees.min()
    ) / (degrees.max() - degrees.min()) * 10.0
else:
    node_sizes = np.full_like(degrees, BASE_NODE_SIZE + 5.0)

node_x, node_y, node_z = [], [], []
node_text, node_color = [], []

for idx, node in enumerate(G.nodes()):
    x, y, z = pos[node]

    degree_val = node_degree[node]
    strength_val = node_strength[node]
    family = get_node_family(node)

    node_x.append(x)
    node_y.append(y)
    node_z.append(z)
    node_color.append(FAMILY_COLORS.get(family, FAMILY_COLORS["unknown"]))

    node_text.append(
        f"node: {node}<br>"
        f"family: {family}<br>"
        f"degree: {degree_val}<br>"
        f"strength(abs precision): {strength_val:.6f}"
    )


# ============================================================
# EDGE TRACES
# ============================================================

print("\nPreparing edge traces...")

abs_weights = np.array(
    [G[u][v]["abs_precision_weight"] for u, v in G.edges()],
    dtype=float
)

w_min = abs_weights.min()
w_max = abs_weights.max()

def scale_width(w: float) -> float:
    if w_max == w_min:
        return (EDGE_WIDTH_MIN + EDGE_WIDTH_MAX) / 2.0
    alpha = (w - w_min) / (w_max - w_min)
    return EDGE_WIDTH_MIN + alpha * (EDGE_WIDTH_MAX - EDGE_WIDTH_MIN)

edge_traces = []

for sign_value, color_value in [("positive", "#1f77b4"), ("negative", "#d62728")]:
    sign_edges = [(u, v) for u, v in G.edges() if G[u][v]["sign"] == sign_value]

    for u, v in sign_edges:
        x0, y0, z0 = pos[u]
        x1, y1, z1 = pos[v]

        abs_w = G[u][v]["abs_precision_weight"]
        p_w = G[u][v]["precision_weight"]
        pc_w = G[u][v]["partial_corr_weight"]

        edge_traces.append(
            go.Scatter3d(
                x=[x0, x1, None],
                y=[y0, y1, None],
                z=[z0, z1, None],
                mode="lines",
                line=dict(
                    color=color_value,
                    width=scale_width(abs_w),
                ),
                opacity=EDGE_OPACITY,
                hoverinfo="text",
                text=[
                    f"{u} ↔ {v}<br>"
                    f"sign: {sign_value}<br>"
                    f"precision_weight: {p_w:.6f}<br>"
                    f"abs_precision_weight: {abs_w:.6f}<br>"
                    f"partial_corr_weight: {pc_w:.6f}",
                    f"{u} ↔ {v}<br>"
                    f"sign: {sign_value}<br>"
                    f"precision_weight: {p_w:.6f}<br>"
                    f"abs_precision_weight: {abs_w:.6f}<br>"
                    f"partial_corr_weight: {pc_w:.6f}",
                    None,
                ],
                showlegend=False,
            )
        )


# ============================================================
# NODE TRACES BY FAMILY
# One trace per family gives a clean legend
# ============================================================

node_traces = []

nodes_list = list(G.nodes())

for family, color in FAMILY_COLORS.items():
    idxs = [
        i for i, node in enumerate(nodes_list)
        if get_node_family(node) == family
    ]

    if len(idxs) == 0:
        continue

    trace = go.Scatter3d(
        x=[node_x[i] for i in idxs],
        y=[node_y[i] for i in idxs],
        z=[node_z[i] for i in idxs],
        mode="markers+text",
        text=[nodes_list[i] for i in idxs],
        textposition="top center",
        hoverinfo="text",
        hovertext=[node_text[i] for i in idxs],
        marker=dict(
            size=[node_sizes[i] for i in idxs],
            color=color,
            opacity=0.95,
            line=dict(width=0.5, color="black"),
        ),
        name=family,
        showlegend=True,
    )

    node_traces.append(trace)


# ============================================================
# FIGURE
# ============================================================

print("\nRendering interactive Plotly figure...")

fig = go.Figure(data=edge_traces + node_traces)

fig.update_layout(
    title=(
        f"{TITLE}<br>"
        f"<sup>nodes={n_nodes}, edges={n_edges}, "
        f"top_k_edges={TOP_K_EDGES}, spring_k={SPRING_K}, iterations={SPRING_ITERATIONS}</sup>"
    ),
    #width=FIG_WIDTH,
    #height=FIG_HEIGHT,
    autosize=True,
    margin=dict(l=0, r=0, b=0, t=60),
    legend=dict(
        title="Node family",
        x=0.02,
        y=0.98,
    ),
    scene=dict(
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
    ),
)

fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn")

print(f"\nSaved HTML to: {OUTPUT_HTML}")
print("\nDONE: 3D graph rendered successfully.")